In [2]:
# ============================================================
# cell- 10
# FAKEDDIT ORIGINAL vs FINAL CLASS COUNTS
# NUMBERS ONLY — TRAIN / VALIDATION / TEST SEPARATELY
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

ORIGINAL = {
    "Train": "/kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv",
    "Validation": "/kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_validate.tsv",
    "Test": "/kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_test_public.tsv",
}

FINAL = {
    "Train": "/kaggle/input/datasets/mariaislambarra/new-124k-data/train_clean_filled_99200.csv",
    "Validation": "/kaggle/input/datasets/mariaislambarra/new-124k-data/val_clean_filled_12400.csv",
    "Test": "/kaggle/input/datasets/mariaislambarra/new-124k-data/test_clean_filled_12400.csv",
}

LABEL_COL = "6_way_label"

CLASS_NAMES = {
    0: "Real",
    1: "Manipulated",
    2: "Misleading",
    3: "Satire",
    4: "False",
    5: "Humorous"
}

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

orig = {
    split: pd.read_csv(
        path,
        sep="\t",
        low_memory=False,
        on_bad_lines="skip"
    )
    for split, path in ORIGINAL.items()
}

final = {
    split: pd.read_csv(
        path,
        low_memory=False
    )
    for split, path in FINAL.items()
}

# ------------------------------------------------------------
# CLASS COUNT FUNCTION
# ------------------------------------------------------------

def get_counts(df):
    labels = pd.to_numeric(
        df[LABEL_COL],
        errors="coerce"
    )

    return (
        labels.value_counts()
        .reindex(range(6), fill_value=0)
        .astype(int)
    )

# ------------------------------------------------------------
# GET COUNTS
# ------------------------------------------------------------

orig_train = get_counts(orig["Train"])
orig_val   = get_counts(orig["Validation"])
orig_test  = get_counts(orig["Test"])

final_train = get_counts(final["Train"])
final_val   = get_counts(final["Validation"])
final_test  = get_counts(final["Test"])

# ------------------------------------------------------------
# CREATE TABLE
# ------------------------------------------------------------

table = pd.DataFrame({
    "Class_ID": range(6),
    "Class": [CLASS_NAMES[i] for i in range(6)],

    "Original_Train": orig_train.values,
    "Final_Train": final_train.values,

    "Original_Validation": orig_val.values,
    "Final_Validation": final_val.values,

    "Original_Test": orig_test.values,
    "Final_Test": final_test.values
})

# Add total row
total_row = pd.DataFrame([{
    "Class_ID": "",
    "Class": "TOTAL",

    "Original_Train": len(orig["Train"]),
    "Final_Train": len(final["Train"]),

    "Original_Validation": len(orig["Validation"]),
    "Final_Validation": len(final["Validation"]),

    "Original_Test": len(orig["Test"]),
    "Final_Test": len(final["Test"])
}])

table_with_total = pd.concat(
    [table, total_row],
    ignore_index=True
)

# ------------------------------------------------------------
# PRINT TABLE
# ------------------------------------------------------------

print("=" * 120)
print("ORIGINAL FAKEDDIT vs FINAL CLEANED DATASET — CLASS COUNTS")
print("=" * 120)

print(table_with_total.to_string(index=False))


for split, original_counts, final_counts in [
    ("TRAIN", orig_train, final_train),
    ("VALIDATION", orig_val, final_val),
    ("TEST", orig_test, final_test)
]:

    split_table = pd.DataFrame({
        "Class_ID": range(6),
        "Class": [CLASS_NAMES[i] for i in range(6)],
        "Original": original_counts.values,
        "Final": final_counts.values
    })

    print("\n" + "=" * 60)
    print(split)
    print("=" * 60)
    print(split_table.to_string(index=False))

    print(
        f"TOTAL: Original = {original_counts.sum():,} | "
        f"Final = {final_counts.sum():,}"
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

table_with_total.to_csv(
    "/kaggle/working/original_vs_final_class_counts.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/original_vs_final_class_counts.csv")

ORIGINAL FAKEDDIT vs FINAL CLEANED DATASET — CLASS COUNTS
Class_ID       Class  Original_Train  Final_Train  Original_Validation  Final_Validation  Original_Test  Final_Test
       0        Real          222081        32106                23320              4062          23507        4123
       1 Manipulated           33481         5323                 3521               694           3514         701
       2  Misleading          107221        19203                11277              2434          11297        2457
       3      Satire           11784         1087                 1238               134           1224         140
       4       False          167857        39222                17810              4818          17472        4682
       5    Humorous           21576         2259                 2176               258           2305         297
               TOTAL          564000        99200                59342             12400          59319       12400

TRAIN
 Class_

# HGR+++ (NO EXPLICIT METADATA CONTROLLED RETRAINING)

In [2]:
# ============================================================
# cell-9
# HGR+++ — NO EXPLICIT METADATA CONTROLLED RETRAINING
#
# PURPOSE:
#   Test HGR+++ after removing ALL explicit metadata.
#
# REMOVED:
#   - score
#   - num_comments
#   - upvote_ratio
#   - hasImage
#   - domain / domain embedding
#
# KEPT:
#   - CLIP text embedding  (512)
#   - CLIP image embedding (512)
#   - compressed CLIP semantic context (128 -> 256)
#   - HGR+++ context-conditioned gate
#   - centered residual modulation
#   - classification head
#
# SAME:
#   Seed = 42
#   Epochs = 6
#   LR = 2e-4
#   Weight decay = 1e-4
#   Label smoothing = 0.02
#   Dropout = 0.2
#   Gate tau = 2
#   Gate lambda = 0.005
#   Gate target = 0.60
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42
EPOCHS = 6

BATCH_TRAIN = 128
BATCH_EVAL = 256

LR = 2e-4
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

DROPOUT = 0.20

GATE_TAU = 2.0
GATE_LAMBDA = 0.005
GATE_TARGET = 0.60

NUM_CLASSES = 6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/"
    "fakeddit1-precomputed-features"
)

SAVE_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_NO_EXPLICIT_METADATA_seed42.pt"
)

RESULT_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_NO_EXPLICIT_METADATA_seed42_results.csv"
)


print("=" * 95)
print("HGR+++ CONTROLLED RETRAINING — NO EXPLICIT METADATA")
print("=" * 95)

print("Device:", DEVICE)
print("Seed  :", SEED)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 3. LOAD ONLY TEXT, IMAGE, LABELS
# ============================================================

def load_tensor(name):

    path = os.path.join(
        FEAT_DIR,
        name
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    return torch.load(
        path,
        map_location="cpu"
    )


def load_split(split):

    return {

        "text":
            load_tensor(
                f"{split}_text.pt"
            ).float(),

        "image":
            load_tensor(
                f"{split}_image.pt"
            ).float(),

        "y":
            load_tensor(
                f"{split}_labels.pt"
            ).long()
    }


train = load_split("train")
val   = load_split("val")
test  = load_split("test")


print("\nDataset sizes:")

print("Train:", len(train["y"]))
print("Val  :", len(val["y"]))
print("Test :", len(test["y"]))


assert len(train["y"]) == 99200
assert len(val["y"]) == 12400
assert len(test["y"]) == 12400

assert train["text"].shape[1] == 512
assert train["image"].shape[1] == 512

print("✓ Dataset sizes verified.")
print("✓ Only CLIP text + image features loaded.")
print("✓ No explicit metadata loaded.")


# ============================================================
# 4. CLASS DISTRIBUTION
# ============================================================

train_counts = torch.bincount(
    train["y"],
    minlength=NUM_CLASSES
).float()


class_weights = (
    len(train["y"])
    /
    (
        NUM_CLASSES
        *
        train_counts
    )
)


print("\nTrain class counts:")

for cls in range(NUM_CLASSES):

    print(
        f"Class {cls}: "
        f"{int(train_counts[cls]):,}"
    )


print(
    "\nClass weights:",
    class_weights.numpy()
)


# ============================================================
# 5. DATASET
# ============================================================

class HGRDatasetNoMetadata(Dataset):

    def __init__(
        self,
        data
    ):

        self.text = data["text"]
        self.image = data["image"]
        self.y = data["y"]


    def __len__(self):

        return len(self.y)


    def __getitem__(
        self,
        idx
    ):

        return (
            self.text[idx],
            self.image[idx],
            self.y[idx]
        )


train_ds = HGRDatasetNoMetadata(
    train
)

val_ds = HGRDatasetNoMetadata(
    val
)

test_ds = HGRDatasetNoMetadata(
    test
)


# ============================================================
# 6. DATA LOADERS
# ============================================================

generator = torch.Generator()
generator.manual_seed(SEED)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_TRAIN,
    shuffle=True,
    generator=generator,
    num_workers=0
)


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_EVAL,
    shuffle=False,
    num_workers=0
)


test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_EVAL,
    shuffle=False,
    num_workers=0
)


# ============================================================
# 7. HGR+++ WITHOUT EXPLICIT METADATA
# =====================================

class HGRPlusPlusNoMetadata(nn.Module):

    def __init__(
        self,
        dropout=0.20,
        gate_tau=2.0
    ):

        super().__init__()

        self.gate_tau = gate_tau


        # ----------------------------------------------------
        # CLIP compression
        # ----------------------------------------------------

        self.clip_compress = nn.Sequential(

            nn.Linear(
                1024,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )



        self.context_mlp = nn.Sequential(

            nn.Linear(
                128,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # ----------------------------------------------------
        # Gate
       # ----------------------------------------------------

        self.gate_logits = nn.Sequential(

            nn.Linear(
                256,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                1280
            )
        )


        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        self.head = nn.Sequential(

            nn.Linear(
                1280,
                1024
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                1024,
                NUM_CLASSES
            )
        )


    def forward(
        self,
        text,
        image
    ):

        # ----------------------------------------------------
        # Normalize CLIP embeddings
        # ----------------------------------------------------

        text = F.normalize(
            text,
            dim=1
        )

        image = F.normalize(
            image,
            dim=1
        )


        # ----------------------------------------------------
        # Raw multimodal representation
        # ----------------------------------------------------

        clip = torch.cat(
            [
                text,
                image
            ],
            dim=1
        )
        # [B, 1024]


        # ----------------------------------------------------
        # Semantic context derived ONLY from CLIP
        # ----------------------------------------------------

        clip_small = self.clip_compress(
            clip
        )
        # [B, 128]


        context = self.context_mlp(
            clip_small
        )
        # [B, 256]


        # ----------------------------------------------------
        # Initial fused representation
        # ----------------------------------------------------

        z0 = torch.cat(
            [
                clip,
                context
            ],
            dim=1
        )
        # [B, 1280]


        # ----------------------------------------------------
        # HGR+++ gate
        # ----------------------------------------------------

        gate = torch.sigmoid(

            self.gate_logits(
                context
            )
            /
            self.gate_tau
        )
        # [B, 1280]


        # ----------------------------------------------------
        # Centered residual modulation
        # ----------------------------------------------------

        z = (
            z0
            +
            z0
            *
            (
                gate
                -
                0.5
            )
        )


        logits = self.head(
            z
        )


        return logits, gate


# ============================================================
# 8. MODEL
# ============================================================

model = HGRPlusPlusNoMetadata(
    dropout=DROPOUT,
    gate_tau=GATE_TAU
).to(DEVICE)


print("\nModel:")
print(model)


# ============================================================
# 9. CLASS-WEIGHTED LABEL-SMOOTHED LOSS
# ============================================================

class_weights_device = class_weights.to(
    DEVICE
)


def weighted_smoothed_ce(
    logits,
    targets
):

    num_classes = logits.size(1)


    with torch.no_grad():

        smooth_target = torch.full_like(
            logits,
            LABEL_SMOOTHING
            /
            (
                num_classes
                -
                1
            )
        )


        smooth_target.scatter_(
            1,
            targets.unsqueeze(1),
            1.0
            -
            LABEL_SMOOTHING
        )


        weighted_target = (
            smooth_target
            *
            class_weights_device.unsqueeze(0)
        )


        weighted_target = (
            weighted_target
            /
            weighted_target.sum(
                dim=1,
                keepdim=True
            )
        )


    log_probs = F.log_softmax(
        logits,
        dim=1
    )


    loss = -(
        weighted_target
        *
        log_probs
    ).sum(
        dim=1
    ).mean()


    return loss


# ============================================================
# 10. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 11. EVALUATION
# ============================================================

def evaluate(
    model,
    loader
):

    model.eval()


    all_true = []
    all_pred = []
    all_prob = []
    all_gate = []


    with torch.no_grad():

        for (
            text,
            image,
            labels
        ) in loader:


            text = text.to(
                DEVICE
            )

            image = image.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )


            logits, gate = model(
                text,
                image
            )


            prob = torch.softmax(
                logits,
                dim=1
            )


            pred = torch.argmax(
                logits,
                dim=1
            )


            all_true.append(
                labels.cpu()
            )

            all_pred.append(
                pred.cpu()
            )

            all_prob.append(
                prob.cpu()
            )

            all_gate.append(
                gate.mean(
                    dim=1
                ).cpu()
            )


    y_true = torch.cat(
        all_true
    ).numpy()


    y_pred = torch.cat(
        all_pred
    ).numpy()


    y_prob = torch.cat(
        all_prob
    ).numpy()


    gate_mean = torch.cat(
        all_gate
    ).numpy()


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    try:

        macro_auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        macro_auc = np.nan


    return {

        "accuracy":
            float(accuracy),

        "macro_precision":
            float(precision),

        "macro_recall":
            float(recall),

        "macro_f1":
            float(macro_f1),

        "weighted_f1":
            float(weighted_f1),

        "macro_auc":
            float(macro_auc),

        "gate_mean":
            float(
                np.mean(
                    gate_mean
                )
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }


# ============================================================
# 12. TRAINING
# ============================================================

best_val_f1 = -1.0
best_epoch = None


print("\n")
print("=" * 95)
print("TRAINING")
print("=" * 95)


for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()


    total_loss = 0.0
    correct = 0
    total = 0


    for (
        text,
        image,
        labels
    ) in train_loader:


        text = text.to(
            DEVICE
        )

        image = image.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits, gate = model(
            text,
            image
        )


        ce_loss = weighted_smoothed_ce(
            logits,
            labels
        )


        # ----------------------------------------------------
        # gate regularization
        # ----------------------------------------------------

        gate_reg = torch.abs(
            gate.mean()
            -
            GATE_TARGET
        )


        loss = (
            ce_loss
            +
            GATE_LAMBDA
            *
            gate_reg
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        total_loss += (
            loss.item()
            *
            labels.size(0)
        )


        pred = logits.argmax(
            dim=1
        )


        correct += (
            pred
            ==
            labels
        ).sum().item()


        total += labels.size(0)


    train_loss = (
        total_loss
        /
        total
    )


    train_acc = (
        correct
        /
        total
    )


    val_metrics = evaluate(
        model,
        val_loader
    )


    print(
        f"\nEpoch {epoch}/{EPOCHS}"
    )

    print(
        f"Train Loss     : "
        f"{train_loss:.6f}"
    )

    print(
        f"Train Accuracy : "
        f"{train_acc:.6f}"
    )

    print(
        f"Val Accuracy   : "
        f"{val_metrics['accuracy']:.6f}"
    )

    print(
        f"Val Macro-F1   : "
        f"{val_metrics['macro_f1']:.6f}"
    )

    print(
        f"Val Macro-AUC  : "
        f"{val_metrics['macro_auc']:.6f}"
    )

    print(
        f"Gate Mean      : "
        f"{val_metrics['gate_mean']:.6f}"
    )


    # --------------------------------------------------------
    # Save best validation Macro-F1
    # --------------------------------------------------------

    if (
        val_metrics[
            "macro_f1"
        ]
        >
        best_val_f1
    ):

        best_val_f1 = (
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch


        torch.save(
            {

                "epoch":
                    epoch,

                "seed":
                    SEED,

                "best_val_macro_f1":
                    best_val_f1,

                "model_state_dict":
                    model.state_dict(),

                "config":
                    {

                        "condition":
                            "NO_EXPLICIT_METADATA",

                        "epochs":
                            EPOCHS,

                        "batch_train":
                            BATCH_TRAIN,

                        "batch_eval":
                            BATCH_EVAL,

                        "lr":
                            LR,

                        "weight_decay":
                            WEIGHT_DECAY,

                        "label_smoothing":
                            LABEL_SMOOTHING,

                        "dropout":
                            DROPOUT,

                        "gate_tau":
                            GATE_TAU,

                        "gate_lambda":
                            GATE_LAMBDA,

                        "gate_target":
                            GATE_TARGET
                    }

            },
            SAVE_PATH
        )


        print(
            "✓ New best checkpoint saved."
        )


# ============================================================
# 13. LOAD BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(
    SAVE_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)


print("\n")
print("=" * 95)
print("BEST CHECKPOINT")
print("=" * 95)

print(
    "Best epoch        :",
    checkpoint[
        "epoch"
    ]
)

print(
    "Best Val Macro-F1 :",
    checkpoint[
        "best_val_macro_f1"
    ]
)


# ============================================================
# 14. FINAL TEST
# ============================================================

test_metrics = evaluate(
    model,
    test_loader
)


print("\n")
print("=" * 95)
print("NO-EXPLICIT-METADATA HGR+++ — TEST RESULTS")
print("=" * 95)


print(
    f"Accuracy        : "
    f"{test_metrics['accuracy']:.6f}"
)

print(
    f"Macro Precision : "
    f"{test_metrics['macro_precision']:.6f}"
)

print(
    f"Macro Recall    : "
    f"{test_metrics['macro_recall']:.6f}"
)

print(
    f"Macro-F1        : "
    f"{test_metrics['macro_f1']:.6f}"
)

print(
    f"Weighted-F1     : "
    f"{test_metrics['weighted_f1']:.6f}"
)

print(
    f"Macro-AUC       : "
    f"{test_metrics['macro_auc']:.6f}"
)

print(
    f"Gate Mean       : "
    f"{test_metrics['gate_mean']:.6f}"
)


# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 95)
print("CLASSIFICATION REPORT")
print("=" * 95)


print(
    classification_report(
        test_metrics[
            "y_true"
        ],
        test_metrics[
            "y_pred"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    test_metrics[
        "y_true"
    ],
    test_metrics[
        "y_pred"
    ]
)


print(
    "\nCONFUSION MATRIX"
)

print(cm)


# ============================================================
# 17. PER-CLASS F1
# ============================================================

per_class_f1 = f1_score(
    test_metrics[
        "y_true"
    ],
    test_metrics[
        "y_pred"
    ],
    average=None,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    zero_division=0
)


print("\n")
print("=" * 95)
print("PER-CLASS F1")
print("=" * 95)


for cls, score in enumerate(
    per_class_f1
):

    print(
        f"Class {cls}: "
        f"{score:.6f}"
    )


# ============================================================
# 18. CONTROLLED COMPARISON
#
# Full checkpoint:
#   Accuracy   0.963790
#   Macro-F1   0.935197
#   Macro-AUC  0.994659
#
# No-domain:
#   Accuracy   0.960403
#   Macro-F1   0.937383
#   Macro-AUC  0.996158
# ============================================================

FULL_ACC = 0.963790
FULL_F1 = 0.935197
FULL_AUC = 0.994659

NO_DOMAIN_ACC = 0.960403
NO_DOMAIN_F1 = 0.937383
NO_DOMAIN_AUC = 0.996158


comparison = pd.DataFrame(
    [
        {
            "Model":
                "Full HGR+++",

            "Accuracy":
                FULL_ACC,

            "Macro_F1":
                FULL_F1,

            "Macro_AUC":
                FULL_AUC
        },

        {
            "Model":
                "HGR+++ No Domain",

            "Accuracy":
                NO_DOMAIN_ACC,

            "Macro_F1":
                NO_DOMAIN_F1,

            "Macro_AUC":
                NO_DOMAIN_AUC
        },

        {
            "Model":
                "HGR+++ No Explicit Metadata",

            "Accuracy":
                test_metrics[
                    "accuracy"
                ],

            "Macro_F1":
                test_metrics[
                    "macro_f1"
                ],

            "Macro_AUC":
                test_metrics[
                    "macro_auc"
                ]
        }
    ]
)


comparison[
    "Delta_Acc_vs_Full"
] = (
    comparison[
        "Accuracy"
    ]
    -
    FULL_ACC
)


comparison[
    "Delta_F1_vs_Full"
] = (
    comparison[
        "Macro_F1"
    ]
    -
    FULL_F1
)


comparison[
    "Delta_AUC_vs_Full"
] = (
    comparison[
        "Macro_AUC"
    ]
    -
    FULL_AUC
)


print("\n")
print("=" * 95)
print("FINAL CONTROLLED COMPARISON")
print("=" * 95)


print(
    comparison.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 19. CLASS-4 SHORTCUT CHECK
# ============================================================

class4_mask = (
    test_metrics[
        "y_true"
    ]
    ==
    4
)


class4_correct = (
    test_metrics[
        "y_pred"
    ][
        class4_mask
    ]
    ==
    4
).sum()


class4_total = (
    class4_mask.sum()
)


class4_recall = (
    class4_correct
    /
    class4_total
)


print("\n")
print("=" * 95)
print("CLASS-4 CHECK AFTER REMOVING ALL EXPLICIT METADATA")
print("=" * 95)


print(
    f"Class-4 total   : "
    f"{class4_total:,}"
)

print(
    f"Correct Class-4 : "
    f"{class4_correct:,}"
)

print(
    f"Class-4 Recall  : "
    f"{class4_recall:.6f}"
)

print(
    f"Class-4 F1      : "
    f"{per_class_f1[4]:.6f}"
)


# ============================================================
# 20. SAVE
# ============================================================

comparison.to_csv(
    RESULT_PATH,
    index=False
)


print("\nSaved checkpoint:")
print(SAVE_PATH)

print("\nSaved results:")
print(RESULT_PATH)

print("\n✓ NO-EXPLICIT-METADATA experiment complete.")

HGR+++ CONTROLLED RETRAINING — NO EXPLICIT METADATA
Device: cuda
Seed  : 42

Dataset sizes:
Train: 99200
Val  : 12400
Test : 12400
✓ Dataset sizes verified.
✓ Only CLIP text + image features loaded.
✓ No explicit metadata loaded.

Train class counts:
Class 0: 32,106
Class 1: 5,323
Class 2: 19,203
Class 3: 1,087
Class 4: 39,222
Class 5: 2,259

Class weights: [ 0.5149609   3.1060178   0.86097664 15.210058    0.42153212  7.318873  ]

Model:
HGRPlusPlusNoMetadata(
  (clip_compress): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (context_mlp): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (gate_logits): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1280, bias=True)
  )
  (head): Sequentia

# HGR+++ NO-DOMAIN CONTROLLED RETRAINING

In [1]:
# ============================================================
# cell-8
# HGR+++ NO-DOMAIN CONTROLLED RETRAINING
#
# PURPOSE:
#   Retrain HGR+++ WITHOUT the categorical domain feature.
#
# KEPT:
#   - CLIP text embedding        (512)
#   - CLIP image embedding       (512)
#   - score
#   - num_comments
#   - upvote_ratio
#   - hasImage
#   - CLIP compressed context
#   - HGR+++ gate
#   - centered residual fusion
#   - same training hyperparameters
#
# REMOVED:
#   - domain categorical feature
#   - domain embedding
#
# This is a controlled ablation.
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42

EPOCHS = 6

BATCH_TRAIN = 128
BATCH_EVAL = 256

LR = 2e-4
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

DROPOUT = 0.20

GATE_TAU = 2.0
GATE_LAMBDA = 0.005
GATE_TARGET = 0.60

NUM_CLASSES = 6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/"
    "fakeddit1-precomputed-features"
)

SAVE_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_NO_DOMAIN_seed42.pt"
)

RESULT_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_NO_DOMAIN_seed42_results.csv"
)


print("=" * 90)
print("HGR+++ CONTROLLED RETRAINING — NO DOMAIN")
print("=" * 90)

print("Device:", DEVICE)
print("Seed  :", SEED)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 3. LOAD TENSORS
# ============================================================

def load_tensor(name):

    path = os.path.join(
        FEAT_DIR,
        name
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    return torch.load(
        path,
        map_location="cpu"
    )


def load_split(split):

    return {
        "text":
            load_tensor(
                f"{split}_text.pt"
            ).float(),

        "image":
            load_tensor(
                f"{split}_image.pt"
            ).float(),

        "y":
            load_tensor(
                f"{split}_labels.pt"
            ).long(),

        "num":
            load_tensor(
                f"{split}_num.pt"
            ).float(),

        "has":
            load_tensor(
                f"{split}_has.pt"
            ).float()
    }


train = load_split("train")
val   = load_split("val")
test  = load_split("test")


print("\nDataset sizes:")

print("Train:", len(train["y"]))
print("Val  :", len(val["y"]))
print("Test :", len(test["y"]))


assert len(train["y"]) == 99200
assert len(val["y"]) == 12400
assert len(test["y"]) == 12400

print("✓ Dataset sizes verified.")


# ============================================================
# 4. CLASS DISTRIBUTION + WEIGHTS
# ============================================================

train_counts = torch.bincount(
    train["y"],
    minlength=NUM_CLASSES
).float()


class_weights = (
    len(train["y"])
    /
    (
        NUM_CLASSES
        *
        train_counts
    )
)


print("\nTrain class counts:")

for cls in range(NUM_CLASSES):

    print(
        f"Class {cls}: "
        f"{int(train_counts[cls]):,}"
    )


print(
    "\nClass weights:",
    class_weights.numpy()
)


# ============================================================
# 5. DATASET
# ============================================================

class HGRDatasetNoDomain(Dataset):

    def __init__(
        self,
        data
    ):

        self.text = data["text"]
        self.image = data["image"]
        self.y = data["y"]
        self.num = data["num"]
        self.has = data["has"]


    def __len__(self):

        return len(self.y)


    def __getitem__(
        self,
        idx
    ):

        return (
            self.text[idx],
            self.image[idx],
            self.y[idx],
            self.num[idx],
            self.has[idx]
        )


train_ds = HGRDatasetNoDomain(train)
val_ds   = HGRDatasetNoDomain(val)
test_ds  = HGRDatasetNoDomain(test)


# ============================================================
# 6. SEEDED DATA LOADERS
# ============================================================

generator = torch.Generator()

generator.manual_seed(SEED)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_TRAIN,
    shuffle=True,
    generator=generator,
    num_workers=0
)


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_EVAL,
    shuffle=False,
    num_workers=0
)


test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_EVAL,
    shuffle=False,
    num_workers=0
)


# ============================================================
# 7. HGR+++ WITHOUT DOMAIN
# ============================================================

class HGRPlusPlusNoDomain(nn.Module):

    def __init__(
        self,
        dropout=0.20,
        gate_tau=2.0
    ):

        super().__init__()

        self.gate_tau = gate_tau


        # --------------------------------------------
        # CLIP compression
        # --------------------------------------------

        self.clip_compress = nn.Sequential(

            nn.Linear(
                1024,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # --------------------------------------------
        # Metadata/context branch

        # --------------------------------------------

        self.meta_mlp = nn.Sequential(

            nn.Linear(
                132,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # --------------------------------------------
        # Gate
        # --------------------------------------------

        self.gate_logits = nn.Sequential(

            nn.Linear(
                256,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                1280
            )
        )


        # --------------------------------------------
        # Classification head
        # --------------------------------------------

        self.head = nn.Sequential(

            nn.Linear(
                1280,
                1024
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                1024,
                NUM_CLASSES
            )
        )


    def forward(
        self,
        text,
        image,
        num,
        has
    ):

        # --------------------------------------------
        # Normalize CLIP embeddings
        # --------------------------------------------

        text = F.normalize(
            text,
            dim=1
        )

        image = F.normalize(
            image,
            dim=1
        )


        # --------------------------------------------
        # Raw multimodal CLIP representation
        # --------------------------------------------

        clip = torch.cat(
            [
                text,
                image
            ],
            dim=1
        )
        # 1024


        # --------------------------------------------
        # Compressed semantic context
        # --------------------------------------------

        clip_small = self.clip_compress(
            clip
        )
        # 128


        # --------------------------------------------
        # Context WITHOUT domain
        # --------------------------------------------

        meta_input = torch.cat(
            [
                num,
                has,
                clip_small
            ],
            dim=1
        )


        context = self.meta_mlp(
            meta_input
        )
        # 256


        # --------------------------------------------
        # Initial fused representation
       # --------------------------------------------

        z0 = torch.cat(
            [
                clip,
                context
            ],
            dim=1
        )


        # --------------------------------------------
        # HGR+++ gate
        # --------------------------------------------

        gate = torch.sigmoid(

            self.gate_logits(
                context
            )
            /
            self.gate_tau

        )


        # --------------------------------------------
        # Centered residual modulation
        # --------------------------------------------

        z = (
            z0
            +
            z0
            *
            (
                gate
                -
                0.5
            )
        )


        logits = self.head(
            z
        )


        return logits, gate


# ============================================================
# 8. INITIALIZE MODEL
# ============================================================

model = HGRPlusPlusNoDomain(
    dropout=DROPOUT,
    gate_tau=GATE_TAU
).to(DEVICE)


print("\nModel:")
print(model)


# ============================================================
# 9. CUSTOM LABEL-SMOOTHED CLASS-WEIGHTED LOSS
# ============================================================

class_weights_device = class_weights.to(
    DEVICE
)


def weighted_smoothed_ce(
    logits,
    targets
):

    num_classes = logits.size(1)


    with torch.no_grad():

        smooth_target = torch.full_like(
            logits,
            LABEL_SMOOTHING
            /
            (
                num_classes
                -
                1
            )
        )


        smooth_target.scatter_(
            1,
            targets.unsqueeze(1),
            1.0
            -
            LABEL_SMOOTHING
        )



        weighted_target = (
            smooth_target
            *
            class_weights_device.unsqueeze(0)
        )


        weighted_target = (
            weighted_target
            /
            weighted_target.sum(
                dim=1,
                keepdim=True
            )
        )


    log_probs = F.log_softmax(
        logits,
        dim=1
    )


    loss = -(
        weighted_target
        *
        log_probs
    ).sum(
        dim=1
    ).mean()


    return loss


# ============================================================
# 10. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 11. EVALUATION FUNCTION
# ============================================================

def evaluate(
    model,
    loader
):

    model.eval()


    all_true = []
    all_pred = []
    all_prob = []
    all_gate = []


    with torch.no_grad():

        for (
            text,
            image,
            labels,
            num,
            has
        ) in loader:


            text = text.to(DEVICE)
            image = image.to(DEVICE)
            labels = labels.to(DEVICE)

            num = num.to(DEVICE)
            has = has.to(DEVICE)


            logits, gate = model(
                text,
                image,
                num,
                has
            )


            prob = torch.softmax(
                logits,
                dim=1
            )


            pred = torch.argmax(
                logits,
                dim=1
            )


            all_true.append(
                labels.cpu()
            )

            all_pred.append(
                pred.cpu()
            )

            all_prob.append(
                prob.cpu()
            )

            all_gate.append(
                gate.mean(
                    dim=1
                ).cpu()
            )


    y_true = torch.cat(
        all_true
    ).numpy()


    y_pred = torch.cat(
        all_pred
    ).numpy()


    y_prob = torch.cat(
        all_prob
    ).numpy()


    gate_mean = torch.cat(
        all_gate
    ).numpy()


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    try:

        macro_auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        macro_auc = np.nan


    return {

        "accuracy":
            float(accuracy),

        "macro_precision":
            float(precision),

        "macro_recall":
            float(recall),

        "macro_f1":
            float(macro_f1),

        "weighted_f1":
            float(weighted_f1),

        "macro_auc":
            float(macro_auc),

        "gate_mean":
            float(
                np.mean(
                    gate_mean
                )
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }


# ============================================================
# 12. TRAIN
#
# Train all 6 epochs.
# Save checkpoint with highest validation Macro-F1.
# ============================================================

best_val_f1 = -1.0
best_epoch = None


print("\n")
print("=" * 90)
print("TRAINING")
print("=" * 90)


for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()


    total_loss = 0.0
    total_ce = 0.0
    total_gate_reg = 0.0

    correct = 0
    total = 0


    for (
        text,
        image,
        labels,
        num,
        has
    ) in train_loader:


        text = text.to(DEVICE)
        image = image.to(DEVICE)
        labels = labels.to(DEVICE)

        num = num.to(DEVICE)
        has = has.to(DEVICE)


        optimizer.zero_grad(
            set_to_none=True
        )


        logits, gate = model(
            text,
            image,
            num,
            has
        )


        ce_loss = weighted_smoothed_ce(
            logits,
            labels
        )


        # Same gate regularization
        gate_reg = torch.abs(
            gate.mean()
            -
            GATE_TARGET
        )


        loss = (
            ce_loss
            +
            GATE_LAMBDA
            *
            gate_reg
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        total_loss += (
            loss.item()
            *
            labels.size(0)
        )


        total_ce += (
            ce_loss.item()
            *
            labels.size(0)
        )


        total_gate_reg += (
            gate_reg.item()
            *
            labels.size(0)
        )


        pred = logits.argmax(
            dim=1
        )


        correct += (
            pred
            ==
            labels
        ).sum().item()


        total += labels.size(0)


    train_loss = (
        total_loss
        /
        total
    )


    train_acc = (
        correct
        /
        total
    )


    val_metrics = evaluate(
        model,
        val_loader
    )


    print(
        f"\nEpoch {epoch}/{EPOCHS}"
    )

    print(
        f"Train Loss     : "
        f"{train_loss:.6f}"
    )

    print(
        f"Train Accuracy : "
        f"{train_acc:.6f}"
    )

    print(
        f"Val Accuracy   : "
        f"{val_metrics['accuracy']:.6f}"
    )

    print(
        f"Val Macro-F1   : "
        f"{val_metrics['macro_f1']:.6f}"
    )

    print(
        f"Val Macro-AUC  : "
        f"{val_metrics['macro_auc']:.6f}"
    )

    print(
        f"Gate Mean      : "
        f"{val_metrics['gate_mean']:.6f}"
    )


    # --------------------------------------------
    # Best validation Macro-F1 checkpoint
    # --------------------------------------------

    if (
        val_metrics[
            "macro_f1"
        ]
        >
        best_val_f1
    ):

        best_val_f1 = (
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch


        torch.save(
            {
                "epoch":
                    epoch,

                "seed":
                    SEED,

                "best_val_macro_f1":
                    best_val_f1,

                "model_state_dict":
                    model.state_dict(),

                "config":
                    {
                        "condition":
                            "NO_DOMAIN",

                        "epochs":
                            EPOCHS,

                        "batch_train":
                            BATCH_TRAIN,

                        "batch_eval":
                            BATCH_EVAL,

                        "lr":
                            LR,

                        "weight_decay":
                            WEIGHT_DECAY,

                        "label_smoothing":
                            LABEL_SMOOTHING,

                        "dropout":
                            DROPOUT,

                        "gate_tau":
                            GATE_TAU,

                        "gate_lambda":
                            GATE_LAMBDA,

                        "gate_target":
                            GATE_TARGET
                    }
            },
            SAVE_PATH
        )


        print(
            "✓ New best checkpoint saved."
        )


# ============================================================
# 13. LOAD BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(
    SAVE_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)


print("\n")
print("=" * 90)
print("BEST CHECKPOINT")
print("=" * 90)

print(
    "Best epoch        :",
    checkpoint["epoch"]
)

print(
    "Best Val Macro-F1 :",
    checkpoint[
        "best_val_macro_f1"
    ]
)


# ============================================================
# 14. FINAL TEST EVALUATION
# ============================================================

test_metrics = evaluate(
    model,
    test_loader
)


print("\n")
print("=" * 90)
print("NO-DOMAIN HGR+++ — TEST RESULTS")
print("=" * 90)


print(
    f"Accuracy        : "
    f"{test_metrics['accuracy']:.6f}"
)

print(
    f"Macro Precision : "
    f"{test_metrics['macro_precision']:.6f}"
)

print(
    f"Macro Recall    : "
    f"{test_metrics['macro_recall']:.6f}"
)

print(
    f"Macro-F1        : "
    f"{test_metrics['macro_f1']:.6f}"
)

print(
    f"Weighted-F1     : "
    f"{test_metrics['weighted_f1']:.6f}"
)

print(
    f"Macro-AUC       : "
    f"{test_metrics['macro_auc']:.6f}"
)

print(
    f"Gate Mean       : "
    f"{test_metrics['gate_mean']:.6f}"
)


# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 90)
print("CLASSIFICATION REPORT")
print("=" * 90)


print(
    classification_report(
        test_metrics[
            "y_true"
        ],
        test_metrics[
            "y_pred"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

print(
    "\nCONFUSION MATRIX"
)

print(
    confusion_matrix(
        test_metrics[
            "y_true"
        ],
        test_metrics[
            "y_pred"
        ]
    )
)


# ============================================================
# 17. COMPARE WITH FULL HGR+++
#
# Existing exact checkpoint:
#
# Accuracy  = 0.963790
# Macro-F1  = 0.935197
# AUC       = 0.994659
# ============================================================

FULL_ACC = 0.963790
FULL_F1  = 0.935197
FULL_AUC = 0.994659


comparison = pd.DataFrame(
    [
        {
            "Model":
                "Full HGR+++",

            "Accuracy":
                FULL_ACC,

            "Macro_F1":
                FULL_F1,

            "Macro_AUC":
                FULL_AUC,

            "Delta_Accuracy":
                0.0,

            "Delta_Macro_F1":
                0.0,

            "Delta_Macro_AUC":
                0.0
        },

        {
            "Model":
                "HGR+++ No Domain",

            "Accuracy":
                test_metrics[
                    "accuracy"
                ],

            "Macro_F1":
                test_metrics[
                    "macro_f1"
                ],

            "Macro_AUC":
                test_metrics[
                    "macro_auc"
                ],

            "Delta_Accuracy":
                (
                    test_metrics[
                        "accuracy"
                    ]
                    -
                    FULL_ACC
                ),

            "Delta_Macro_F1":
                (
                    test_metrics[
                        "macro_f1"
                    ]
                    -
                    FULL_F1
                ),

            "Delta_Macro_AUC":
                (
                    test_metrics[
                        "macro_auc"
                    ]
                    -
                    FULL_AUC
                )
        }
    ]
)


print("\n")
print("=" * 90)
print("CONTROLLED COMPARISON")
print("=" * 90)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 18. SAVE RESULTS
# ============================================================

comparison.to_csv(
    RESULT_PATH,
    index=False
)


print("\nSaved checkpoint:")
print(SAVE_PATH)

print("\nSaved comparison:")
print(RESULT_PATH)

HGR+++ CONTROLLED RETRAINING — NO DOMAIN
Device: cuda
Seed  : 42

Dataset sizes:
Train: 99200
Val  : 12400
Test : 12400
✓ Dataset sizes verified.

Train class counts:
Class 0: 32,106
Class 1: 5,323
Class 2: 19,203
Class 3: 1,087
Class 4: 39,222
Class 5: 2,259

Class weights: [ 0.5149609   3.1060178   0.86097664 15.210058    0.42153212  7.318873  ]

Model:
HGRPlusPlusNoDomain(
  (clip_compress): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (meta_mlp): Sequential(
    (0): Linear(in_features=132, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (gate_logits): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1280, bias=True)
  )
  (head): Sequential(
    (0): Linear(in_features=1280, out_features=1024, bias=True)
    (1): ReLU()
    (2

In [15]:
# ============================================================
# ccell- 7
# HGR+++ CROSS-SPLIT METADATA ARTIFACT AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Check Train / Validation / Test for:
#
# 1. Numeric metadata distributions by class
# 2. Domain -> label relationships
# 3. Special investigation of domain ID 21
# 4. Special investigation of Class 4
# 5. Whether metadata values are constant within classes
# 6. Whether domains nearly/deterministically identify labels
# 7. Whether the same patterns exist across all splits
#
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

# ============================================================
# 1. CONFIG
# ============================================================

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/fakeddit1-precomputed-features"
)

OUTPUT_DIR = "/kaggle/working"

NUMERIC_NAMES = [
    "score",
    "num_comments",
    "upvote_ratio"
]

print("=" * 95)
print("HGR+++ CROSS-SPLIT METADATA ARTIFACT AUDIT")
print("=" * 95)


# ============================================================
# 2. LOAD FUNCTION
# ============================================================

def load_tensor(filename):

    path = os.path.join(
        FEAT_DIR,
        filename
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file: {path}"
        )

    return torch.load(
        path,
        map_location="cpu"
    )


# ============================================================
# 3. LOAD TRAIN / VAL / TEST
# ============================================================

splits = {}

for split in [
    "train",
    "val",
    "test"
]:

    splits[split] = {

        "y":
            load_tensor(
                f"{split}_labels.pt"
            ).long(),

        "num":
            load_tensor(
                f"{split}_num.pt"
            ).float(),

        "has":
            load_tensor(
                f"{split}_has.pt"
            ).float(),

        "cat":
            load_tensor(
                f"{split}_cat.pt"
            ).long()
    }


print("\nLoaded splits:")

for split, data in splits.items():

    print(
        f"{split:<6}: "
        f"N={len(data['y']):,} | "
        f"num={tuple(data['num'].shape)} | "
        f"has={tuple(data['has'].shape)} | "
        f"cat={tuple(data['cat'].shape)}"
    )


# ============================================================
# 4. VERIFY EXPECTED SIZES
# ============================================================

expected_sizes = {
    "train": 99200,
    "val":   12400,
    "test":  12400
}

for split in expected_sizes:

    assert (
        len(splits[split]["y"])
        ==
        expected_sizes[split]
    )

print(
    "\n✓ Split sizes verified."
)


# ============================================================
# 5. CLASS DISTRIBUTIONS
# ============================================================

print("\n")
print("=" * 95)
print("CLASS DISTRIBUTIONS")
print("=" * 95)


for split, data in splits.items():

    y = data[
        "y"
    ].numpy()

    labels, counts = np.unique(
        y,
        return_counts=True
    )

    print(
        f"\n{split.upper()}"
    )

    for label, count in zip(
        labels,
        counts
    ):

        print(
            f"Class {int(label)}: "
            f"{int(count):,}"
        )


# ============================================================
# 6. NUMERIC METADATA BY CLASS AND SPLIT
# ============================================================

numeric_rows = []


print("\n")
print("=" * 95)
print("NUMERIC METADATA BY CLASS ACROSS SPLITS")
print("=" * 95)


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    num = (
        data["num"]
        .numpy()
    )

    print(
        f"\n{'-' * 95}"
    )

    print(
        split.upper()
    )

    print(
        "-" * 95
    )


    for cls in sorted(
        np.unique(y)
    ):

        mask = (
            y == cls
        )

        print(
            f"\nClass {int(cls)} "
            f"| N={mask.sum():,}"
        )


        for j, feature in enumerate(
            NUMERIC_NAMES
        ):

            values = num[
                mask,
                j
            ]

            mean = float(
                np.mean(values)
            )

            std = float(
                np.std(values)
            )

            median = float(
                np.median(values)
            )

            min_val = float(
                np.min(values)
            )

            max_val = float(
                np.max(values)
            )

            unique_count = int(
                len(
                    np.unique(values)
                )
            )


            numeric_rows.append(
                {
                    "split":
                        split,

                    "class":
                        int(cls),

                    "feature":
                        feature,

                    "n":
                        int(mask.sum()),

                    "mean":
                        mean,

                    "std":
                        std,

                    "median":
                        median,

                    "min":
                        min_val,

                    "max":
                        max_val,

                    "unique_values":
                        unique_count
                }
            )


            print(
                f"{feature:<16}"
                f" mean={mean:>9.4f}"
                f" std={std:>9.4f}"
                f" unique={unique_count:>6}"
            )


numeric_df = pd.DataFrame(
    numeric_rows
)


# ============================================================
# 7. FIND CONSTANT NUMERIC FEATURES WITHIN A CLASS
# ============================================================

print("\n")
print("=" * 95)
print("CONSTANT NUMERIC FEATURES WITHIN CLASSES")
print("=" * 95)


constant_df = numeric_df[
    numeric_df[
        "unique_values"
    ]
    ==
    1
].copy()


if len(constant_df) == 0:

    print(
        "No class has a completely constant "
        "numeric metadata feature."
    )

else:

    print(
        constant_df[
            [
                "split",
                "class",
                "feature",
                "n",
                "mean",
                "unique_values"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 8. hasImage BY SPLIT + CLASS
# ============================================================

print("\n")
print("=" * 95)
print("hasImage BY CLASS ACROSS SPLITS")
print("=" * 95)


has_rows = []


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    has = (
        data["has"]
        .numpy()
        .reshape(-1)
    )

    print(
        f"\n{split.upper()}"
    )


    for cls in sorted(
        np.unique(y)
    ):

        mask = (
            y == cls
        )

        values = has[
            mask
        ]

        unique_values = np.unique(
            values
        )

        mean_value = float(
            np.mean(values)
        )


        has_rows.append(
            {
                "split":
                    split,

                "class":
                    int(cls),

                "n":
                    int(mask.sum()),

                "mean_hasImage":
                    mean_value,

                "unique_values":
                    len(unique_values)
            }
        )


        print(
            f"Class {int(cls)}: "
            f"N={mask.sum():,} | "
            f"mean={mean_value:.6f} | "
            f"unique={len(unique_values)}"
        )


has_df = pd.DataFrame(
    has_rows
)


# ============================================================
# 9. DOMAIN-LABEL ANALYSIS FOR EACH SPLIT
# ============================================================

domain_all_rows = []


print("\n")
print("=" * 95)
print("DOMAIN -> LABEL RELATIONSHIPS ACROSS SPLITS")
print("=" * 95)


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    domain = (
        data["cat"][:, 0]
        .numpy()
    )


    frame = pd.DataFrame(
        {
            "domain_id":
                domain,

            "label":
                y
        }
    )


    print(
        f"\n{split.upper()}"
    )

    print(
        "Unique domains:",
        frame[
            "domain_id"
        ].nunique()
    )


    domain_counts = (
        frame
        .groupby(
            "domain_id"
        )
        .size()
    )


    label_counts = (
        frame
        .groupby(
            [
                "domain_id",
                "label"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )


    split_rows = []


    for domain_id, total in (
        domain_counts.items()
    ):

        temp = label_counts[
            label_counts[
                "domain_id"
            ]
            ==
            domain_id
        ].sort_values(
            "count",
            ascending=False
        )


        top = temp.iloc[
            0
        ]


        dominant_label = int(
            top[
                "label"
            ]
        )

        dominant_count = int(
            top[
                "count"
            ]
        )

        purity = (
            dominant_count
            /
            int(total)
        )


        row = {

            "split":
                split,

            "domain_id":
                int(domain_id),

            "total":
                int(total),

            "dominant_label":
                dominant_label,

            "dominant_count":
                dominant_count,

            "purity":
                float(purity),

            "num_labels_present":
                int(
                    temp[
                        "label"
                    ].nunique()
                )
        }


        split_rows.append(
            row
        )

        domain_all_rows.append(
            row
        )


    split_summary = pd.DataFrame(
        split_rows
    ).sort_values(
        "total",
        ascending=False
    )


    print(
        "\nTop 15 domains:"
    )

    print(
        split_summary
        .head(15)
        .to_string(
            index=False
        )
    )


domain_df = pd.DataFrame(
    domain_all_rows
)


# ============================================================
# 10. SPECIAL INVESTIGATION:
#     DOMAIN ID 21
# ============================================================

print("\n")
print("=" * 95)
print("SPECIAL CHECK: DOMAIN ID 21")
print("=" * 95)


domain21_rows = []


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    domain = (
        data["cat"][:, 0]
        .numpy()
    )

    mask = (
        domain == 21
    )

    labels = y[
        mask
    ]


    print(
        f"\n{split.upper()}"
    )

    print(
        f"Domain 21 samples: "
        f"{mask.sum():,}"
    )


    if mask.sum() == 0:

        print(
            "No domain-21 samples."
        )

        continue


    unique_labels, counts = np.unique(
        labels,
        return_counts=True
    )


    for label, count in zip(
        unique_labels,
        counts
    ):

        percentage = (
            count
            /
            mask.sum()
            *
            100
        )

        print(
            f"Class {int(label)}: "
            f"{int(count):,} "
            f"({percentage:.4f}%)"
        )


        domain21_rows.append(
            {
                "split":
                    split,

                "domain_id":
                    21,

                "class":
                    int(label),

                "count":
                    int(count),

                "percentage":
                    float(
                        percentage
                    )
            }
        )


# ============================================================
# 11. SPECIAL INVESTIGATION:
#     CLASS 4 DOMAIN DISTRIBUTION
# ============================================================

print("\n")
print("=" * 95)
print("SPECIAL CHECK: CLASS 4 DOMAIN DISTRIBUTION")
print("=" * 95)


class4_domain_rows = []


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    domain = (
        data["cat"][:, 0]
        .numpy()
    )


    class_mask = (
        y == 4
    )

    class_domains = domain[
        class_mask
    ]


    unique_domains, counts = np.unique(
        class_domains,
        return_counts=True
    )


    order = np.argsort(
        counts
    )[::-1]


    print(
        f"\n{split.upper()}"
    )

    print(
        f"Class-4 samples: "
        f"{class_mask.sum():,}"
    )

    print(
        "Unique domains:",
        len(
            unique_domains
        )
    )


    for idx in order[:20]:

        domain_id = int(
            unique_domains[
                idx
            ]
        )

        count = int(
            counts[
                idx
            ]
        )

        pct = (
            count
            /
            class_mask.sum()
            *
            100
        )


        print(
            f"Domain {domain_id:>4}: "
            f"{count:>6,} "
            f"({pct:>8.4f}%)"
        )


        class4_domain_rows.append(
            {
                "split":
                    split,

                "domain_id":
                    domain_id,

                "count":
                    count,

                "percentage_of_class4":
                    pct
            }
        )


# ============================================================
# 12. CLASS 4 NUMERIC METADATA — DETAILED
# ============================================================

print("\n")
print("=" * 95)
print("SPECIAL CHECK: CLASS 4 NUMERIC METADATA")
print("=" * 95)


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    num = (
        data["num"]
        .numpy()
    )


    mask = (
        y == 4
    )


    print(
        f"\n{split.upper()} "
        f"| Class-4 N={mask.sum():,}"
    )


    for j, feature in enumerate(
        NUMERIC_NAMES
    ):

        values = num[
            mask,
            j
        ]


        unique_values = np.unique(
            values
        )


        print(
            f"\n{feature}"
        )

        print(
            "  unique values:",
            len(
                unique_values
            )
        )

        print(
            "  mean:",
            float(
                np.mean(
                    values
                )
            )
        )

        print(
            "  std :",
            float(
                np.std(
                    values
                )
            )
        )


        if len(
            unique_values
        ) <= 10:

            print(
                "  values:",
                unique_values
            )


# ============================================================
# 13. CHECK WHETHER DOMAIN 21 AND CLASS 4 ARE
#     EXACTLY EQUIVALENT
# ============================================================

print("\n")
print("=" * 95)
print("DOMAIN 21 <-> CLASS 4 EQUIVALENCE CHECK")
print("=" * 95)


equivalence_rows = []


for split, data in splits.items():

    y = (
        data["y"]
        .numpy()
    )

    domain = (
        data["cat"][:, 0]
        .numpy()
    )


    domain21 = (
        domain == 21
    )

    class4 = (
        y == 4
    )


    # P(Class 4 | Domain 21)

    if domain21.sum() > 0:

        p_class4_given_domain21 = (
            (
                class4
                &
                domain21
            ).sum()
            /
            domain21.sum()
        )

    else:

        p_class4_given_domain21 = np.nan


    # P(Domain 21 | Class 4)

    if class4.sum() > 0:

        p_domain21_given_class4 = (
            (
                class4
                &
                domain21
            ).sum()
            /
            class4.sum()
        )

    else:

        p_domain21_given_class4 = np.nan


    exact_equivalent = bool(
        np.array_equal(
            domain21,
            class4
        )
    )


    equivalence_rows.append(
        {
            "split":
                split,

            "domain21_N":
                int(
                    domain21.sum()
                ),

            "class4_N":
                int(
                    class4.sum()
                ),

            "P(Class4|Domain21)":
                p_class4_given_domain21,

            "P(Domain21|Class4)":
                p_domain21_given_class4,

            "exact_equivalence":
                exact_equivalent
        }
    )


    print(
        f"\n{split.upper()}"
    )

    print(
        f"Domain-21 N : "
        f"{domain21.sum():,}"
    )

    print(
        f"Class-4 N   : "
        f"{class4.sum():,}"
    )

    print(
        "P(Class 4 | Domain 21) = "
        f"{p_class4_given_domain21:.6f}"
    )

    print(
        "P(Domain 21 | Class 4) = "
        f"{p_domain21_given_class4:.6f}"
    )

    print(
        "Exact equivalence       =",
        exact_equivalent
    )


equivalence_df = pd.DataFrame(
    equivalence_rows
)


# ============================================================
# 14. LARGE HIGH-PURITY DOMAINS
#
# We focus on domains with meaningful support.
# ============================================================

print("\n")
print("=" * 95)
print("HIGH-PURITY DOMAINS WITH SUPPORT >= 20")
print("=" * 95)


high_purity = domain_df[
    (
        domain_df[
            "total"
        ]
        >=
        20
    )
    &
    (
        domain_df[
            "purity"
        ]
        >=
        0.90
    )
].copy()


if len(
    high_purity
) == 0:

    print(
        "No domains meet the criteria."
    )

else:

    print(
        high_purity
        .sort_values(
            [
                "split",
                "total"
            ],
            ascending=[
                True,
                False
            ]
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 15. SAVE ALL RESULTS
# ============================================================

numeric_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_split_numeric_metadata.csv"
    ),
    index=False
)


constant_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_split_constant_metadata.csv"
    ),
    index=False
)


has_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_split_hasImage.csv"
    ),
    index=False
)


domain_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_split_domain_label_purity.csv"
    ),
    index=False
)


pd.DataFrame(
    domain21_rows
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "domain21_label_distribution.csv"
    ),
    index=False
)


pd.DataFrame(
    class4_domain_rows
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "class4_domain_distribution.csv"
    ),
    index=False
)


equivalence_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "domain21_class4_equivalence.csv"
    ),
    index=False
)


high_purity.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "high_purity_domains_support20.csv"
    ),
    index=False
)


# ============================================================
# 16. FINAL COMPACT SUMMARY
# ============================================================

print("\n")
print("=" * 95)
print("FINAL DOMAIN-21 / CLASS-4 SUMMARY")
print("=" * 95)

print(
    equivalence_df.to_string(
        index=False
    )
)


print("\n")
print("=" * 95)
print("FILES SAVED")
print("=" * 95)

for filename in [

    "cross_split_numeric_metadata.csv",
    "cross_split_constant_metadata.csv",
    "cross_split_hasImage.csv",
    "cross_split_domain_label_purity.csv",
    "domain21_label_distribution.csv",
    "class4_domain_distribution.csv",
    "domain21_class4_equivalence.csv",
    "high_purity_domains_support20.csv"

]:

    print(
        os.path.join(
            OUTPUT_DIR,
            filename
        )
    )

HGR+++ CROSS-SPLIT METADATA ARTIFACT AUDIT

Loaded splits:
train : N=99,200 | num=(99200, 3) | has=(99200, 1) | cat=(99200, 1)
val   : N=12,400 | num=(12400, 3) | has=(12400, 1) | cat=(12400, 1)
test  : N=12,400 | num=(12400, 3) | has=(12400, 1) | cat=(12400, 1)

✓ Split sizes verified.


CLASS DISTRIBUTIONS

TRAIN
Class 0: 32,106
Class 1: 5,323
Class 2: 19,203
Class 3: 1,087
Class 4: 39,222
Class 5: 2,259

VAL
Class 0: 4,062
Class 1: 694
Class 2: 2,434
Class 3: 134
Class 4: 4,818
Class 5: 258

TEST
Class 0: 4,123
Class 1: 701
Class 2: 2,457
Class 3: 140
Class 4: 4,682
Class 5: 297


NUMERIC METADATA BY CLASS ACROSS SPLITS

-----------------------------------------------------------------------------------------------
TRAIN
-----------------------------------------------------------------------------------------------

Class 0 | N=32,106
score            mean=   0.0023 std=   0.9246 unique=  1751
num_comments     mean=   0.7270 std=   0.9790 unique=   600
upvote_ratio     mean=   0.724

In [14]:
# ============================================================
# cell- 6
# HGR+++ METADATA SHORTCUT / ARTIFACT AUDIT
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score
)

# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42
BATCH_SIZE = 256

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

OUTPUT_CSV = (
    "/kaggle/working/"
    "HGR_plus_plus_metadata_sensitivity.csv"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 90)
print("HGR+++ METADATA SHORTCUT / ARTIFACT AUDIT")
print("=" * 90)

print("Device:", DEVICE)
print("Seed  :", SEED)


# ============================================================
# 2. VERIFY REQUIRED OBJECTS EXIST
# ============================================================

required_names = [
    "model_exact",
    "test_text",
    "test_image",
    "test_y",
    "test_num",
    "test_has",
    "test_cat"
]

for name in required_names:

    if name not in globals():

        raise RuntimeError(
            f"{name} is missing.\n"
            "Run the previous exact-checkpoint "
            "sensitivity cell first."
        )

print(
    "\n✓ Exact saved HGR+++ model and "
    "test tensors are available."
)


# ============================================================
# 3. BASIC SHAPES
# ============================================================

print("\nTensor shapes:")

print("Text   :", tuple(test_text.shape))
print("Image  :", tuple(test_image.shape))
print("Labels :", tuple(test_y.shape))
print("Numeric:", tuple(test_num.shape))
print("HasImg :", tuple(test_has.shape))
print("Domain :", tuple(test_cat.shape))

assert len(test_y) == 12400


# ============================================================
# 4. NUMERIC METADATA DISTRIBUTION BY CLASS
# ============================================================

NUMERIC_NAMES = [
    "score",
    "num_comments",
    "upvote_ratio"
]

num_np = (
    test_num
    .cpu()
    .numpy()
)

y_np = (
    test_y
    .cpu()
    .numpy()
)


print("\n")
print("=" * 90)
print("NUMERIC METADATA BY CLASS")
print("=" * 90)


numeric_rows = []

for cls in sorted(
    np.unique(y_np)
):

    mask = (
        y_np == cls
    )

    print(
        f"\nClass {cls} | "
        f"N = {mask.sum():,}"
    )

    for j, name in enumerate(
        NUMERIC_NAMES
    ):

        values = num_np[
            mask,
            j
        ]

        row = {
            "class":
                int(cls),

            "feature":
                name,

            "n":
                int(len(values)),

            "mean":
                float(
                    np.mean(values)
                ),

            "std":
                float(
                    np.std(values)
                ),

            "median":
                float(
                    np.median(values)
                ),

            "min":
                float(
                    np.min(values)
                ),

            "max":
                float(
                    np.max(values)
                )
        }

        numeric_rows.append(
            row
        )

        print(
            f"{name:<16}"
            f" mean={row['mean']:>9.4f}"
            f" std={row['std']:>9.4f}"
            f" median={row['median']:>9.4f}"
        )


numeric_df = pd.DataFrame(
    numeric_rows
)

numeric_df.to_csv(
    "/kaggle/working/"
    "metadata_numeric_by_class.csv",
    index=False
)


# ============================================================
# 5. hasImage DISTRIBUTION BY CLASS
# ============================================================

has_np = (
    test_has
    .cpu()
    .numpy()
    .reshape(-1)
)


print("\n")
print("=" * 90)
print("hasImage DISTRIBUTION BY CLASS")
print("=" * 90)


has_rows = []

for cls in sorted(
    np.unique(y_np)
):

    mask = (
        y_np == cls
    )

    values = has_np[
        mask
    ]

    mean_has = float(
        np.mean(values)
    )

    has_rows.append(
        {
            "class":
                int(cls),

            "n":
                int(mask.sum()),

            "mean_hasImage":
                mean_has
        }
    )

    print(
        f"Class {cls}: "
        f"N={mask.sum():,} | "
        f"mean hasImage={mean_has:.6f}"
    )


has_df = pd.DataFrame(
    has_rows
)

has_df.to_csv(
    "/kaggle/working/"
    "metadata_hasImage_by_class.csv",
    index=False
)


# ============================================================
# 6. DOMAIN ↔ LABEL ASSOCIATION
#
# We examine:
#
#   - number of unique domains
#   - most frequent domains
#   - dominant label per domain
#   - label purity
#
# Purity:
#
# max label count for domain
# --------------------------
# total samples for domain
#
# High purity does not automatically mean leakage.
# ============================================================

domain_np = (
    test_cat[:, 0]
    .cpu()
    .numpy()
)


domain_frame = pd.DataFrame(
    {
        "domain_id":
            domain_np,

        "label":
            y_np
    }
)


domain_counts = (
    domain_frame
    .groupby(
        "domain_id"
    )
    .size()
    .rename(
        "total"
    )
)


domain_label_counts = (
    domain_frame
    .groupby(
        [
            "domain_id",
            "label"
        ]
    )
    .size()
    .rename(
        "count"
    )
    .reset_index()
)


domain_summary_rows = []


for domain_id, total in (
    domain_counts.items()
):

    temp = (
        domain_label_counts[
            domain_label_counts[
                "domain_id"
            ]
            ==
            domain_id
        ]
        .sort_values(
            "count",
            ascending=False
        )
    )

    dominant = temp.iloc[0]

    dominant_label = int(
        dominant[
            "label"
        ]
    )

    dominant_count = int(
        dominant[
            "count"
        ]
    )

    purity = (
        dominant_count
        /
        int(total)
    )

    domain_summary_rows.append(
        {
            "domain_id":
                int(domain_id),

            "total":
                int(total),

            "dominant_label":
                dominant_label,

            "dominant_count":
                dominant_count,

            "purity":
                float(purity),

            "num_labels_present":
                int(
                    temp[
                        "label"
                    ].nunique()
                )
        }
    )


domain_summary = (
    pd.DataFrame(
        domain_summary_rows
    )
    .sort_values(
        [
            "total",
            "purity"
        ],
        ascending=[
            False,
            False
        ]
    )
)


print("\n")
print("=" * 90)
print("DOMAIN ↔ LABEL ANALYSIS")
print("=" * 90)

print(
    "Unique test domains:",
    domain_frame[
        "domain_id"
    ].nunique()
)

print(
    "\nTop 30 domains by test frequency:"
)

print(
    domain_summary
    .head(30)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. DOMAIN PURITY — WITH SUPPORT THRESHOLDS
#
# Important because a domain occurring once has purity=1.0
# ============================================================

print("\n")
print("=" * 90)
print("DOMAIN PURITY SUMMARY")
print("=" * 90)


for min_support in [
    5,
    10,
    20,
    50,
    100
]:

    supported = domain_summary[
        domain_summary[
            "total"
        ]
        >=
        min_support
    ]

    if len(supported) == 0:

        continue

    high90 = (
        supported[
            "purity"
        ]
        >=
        0.90
    ).sum()

    pure100 = (
        supported[
            "purity"
        ]
        ==
        1.0
    ).sum()

    print(
        f"Domains with support >= {min_support:>3}: "
        f"{len(supported):>4} | "
        f"purity >= .90: {high90:>4} | "
        f"purity = 1.00: {pure100:>4}"
    )


domain_summary.to_csv(
    "/kaggle/working/"
    "metadata_domain_label_purity.csv",
    index=False
)


# ============================================================
# 8. CREATE PERTURBATION DATASET
# ============================================================

class PerturbationDataset(Dataset):

    def __init__(
        self,
        text,
        image,
        labels,
        num,
        has,
        cat
    ):

        self.text = text
        self.image = image
        self.labels = labels
        self.num = num
        self.has = has
        self.cat = cat


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.text[idx],
            self.image[idx],
            self.labels[idx],
            self.num[idx],
            self.has[idx],
            self.cat[idx]
        )


# ============================================================
# 9. EVALUATION FUNCTION
# ============================================================

def evaluate_metadata_condition(
    condition_name,
    num_tensor,
    has_tensor,
    cat_tensor
):

    dataset = PerturbationDataset(
        test_text,
        test_image,
        test_y,
        num_tensor,
        has_tensor,
        cat_tensor
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    all_true = []
    all_pred = []
    all_prob = []
    all_gate_mean = []


    model_exact.eval()


    with torch.no_grad():

        for (
            text,
            image,
            labels,
            num,
            has,
            cat
        ) in loader:


            text = text.to(
                DEVICE
            )

            image = image.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )

            num = num.to(
                DEVICE
            )

            has = has.to(
                DEVICE
            )

            cat = cat.to(
                DEVICE
            )


            logits, gate = model_exact(
                text,
                image,
                num,
                has,
                cat
            )


            prob = torch.softmax(
                logits,
                dim=1
            )

            pred = torch.argmax(
                logits,
                dim=1
            )


            all_true.append(
                labels.cpu()
            )

            all_pred.append(
                pred.cpu()
            )

            all_prob.append(
                prob.cpu()
            )

            all_gate_mean.append(
                gate.mean(
                    dim=1
                ).cpu()
            )


    y_true = torch.cat(
        all_true
    ).numpy()

    y_pred = torch.cat(
        all_pred
    ).numpy()

    y_prob = torch.cat(
        all_prob
    ).numpy()

    gate_means = torch.cat(
        all_gate_mean
    ).numpy()


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    try:

        auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        auc = np.nan


    return {
        "Condition":
            condition_name,

        "Accuracy":
            accuracy,

        "Macro_Precision":
            precision,

        "Macro_Recall":
            recall,

        "Macro_F1":
            macro_f1,

        "Weighted_F1":
            weighted_f1,

        "Macro_AUC":
            auc,

        "Gate_Mean":
            float(
                np.mean(
                    gate_means
                )
            ),

        "Gate_Std":
            float(
                np.std(
                    gate_means
                )
            )
    }


# ============================================================
# 10. FIXED RANDOM PERMUTATIONS
# ============================================================

generator = torch.Generator()

generator.manual_seed(
    SEED
)

N = len(
    test_y
)


perm_num = torch.randperm(
    N,
    generator=generator
)

perm_has = torch.randperm(
    N,
    generator=generator
)

perm_cat = torch.randperm(
    N,
    generator=generator
)


# ============================================================
# 11. NORMAL METADATA
# ============================================================

results = []


normal = evaluate_metadata_condition(
    "Normal metadata",
    test_num,
    test_has,
    test_cat
)

results.append(
    normal
)


# ============================================================
# 12. SHUFFLE NUMERIC METADATA ONLY
#
# score + num_comments + upvote_ratio
# are moved between test examples.
# ============================================================

numeric_shuffled = evaluate_metadata_condition(
    "Shuffle numeric",
    test_num[
        perm_num
    ],
    test_has,
    test_cat
)

results.append(
    numeric_shuffled
)


# ============================================================
# 13. SHUFFLE hasImage ONLY
# ============================================================

has_shuffled = evaluate_metadata_condition(
    "Shuffle hasImage",
    test_num,
    test_has[
        perm_has
    ],
    test_cat
)

results.append(
    has_shuffled
)


# ============================================================
# 14. SHUFFLE DOMAIN ONLY
# ============================================================

domain_shuffled = evaluate_metadata_condition(
    "Shuffle domain",
    test_num,
    test_has,
    test_cat[
        perm_cat
    ]
)

results.append(
    domain_shuffled
)


# ============================================================
# 15. SHUFFLE ALL METADATA
# ============================================================

all_shuffled = evaluate_metadata_condition(
    "Shuffle all metadata",
    test_num[
        perm_num
    ],
    test_has[
        perm_has
    ],
    test_cat[
        perm_cat
    ]
)

results.append(
    all_shuffled
)


# ============================================================
# 16. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)


baseline_acc = normal[
    "Accuracy"
]

baseline_f1 = normal[
    "Macro_F1"
]

baseline_auc = normal[
    "Macro_AUC"
]


results_df[
    "Accuracy_Delta"
] = (
    results_df[
        "Accuracy"
    ]
    -
    baseline_acc
)


results_df[
    "Macro_F1_Delta"
] = (
    results_df[
        "Macro_F1"
    ]
    -
    baseline_f1
)


results_df[
    "Macro_AUC_Delta"
] = (
    results_df[
        "Macro_AUC"
    ]
    -
    baseline_auc
)


# ============================================================
# 17. PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 115)
print("METADATA PERTURBATION SENSITIVITY RESULTS")
print("=" * 115)


print(
    f"{'Condition':<24}"
    f"{'Accuracy':>12}"
    f"{'Δ Acc':>11}"
    f"{'Macro-F1':>12}"
    f"{'Δ F1':>11}"
    f"{'AUC':>12}"
    f"{'Δ AUC':>11}"
    f"{'Gate Mean':>12}"
)

print("-" * 115)


for _, row in results_df.iterrows():

    print(
        f"{row['Condition']:<24}"
        f"{row['Accuracy']:>12.4f}"
        f"{row['Accuracy_Delta']:>+11.4f}"
        f"{row['Macro_F1']:>12.4f}"
        f"{row['Macro_F1_Delta']:>+11.4f}"
        f"{row['Macro_AUC']:>12.4f}"
        f"{row['Macro_AUC_Delta']:>+11.4f}"
        f"{row['Gate_Mean']:>12.4f}"
    )


# ============================================================
# 18. SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)


print("\nSaved:")
print(
    OUTPUT_CSV
)


print(
    "\nAdditional files:"
)

print(
    "/kaggle/working/"
    "metadata_numeric_by_class.csv"
)

print(
    "/kaggle/working/"
    "metadata_hasImage_by_class.csv"
)

print(
    "/kaggle/working/"
    "metadata_domain_label_purity.csv"
)


# ============================================================
# 19. SANITY CHECK
# ============================================================

if (
    abs(
        normal[
            "Accuracy"
        ]
        -
        0.9638
    )
    >
    0.002
    or
    abs(
        normal[
            "Macro_F1"
        ]
        -
        0.9352
    )
    >
    0.002
):

    print(
        "\nWARNING:"
    )

    print(
        "Normal-metadata performance does not "
        "match the previous checkpoint result."
    )

    print(
        "Do NOT interpret perturbation results yet."
    )

else:

    print(
        "\n✓ Normal-metadata checkpoint "
        "performance reproduced."
    )

HGR+++ METADATA SHORTCUT / ARTIFACT AUDIT
Device: cuda
Seed  : 42

✓ Exact saved HGR+++ model and test tensors are available.

Tensor shapes:
Text   : (12400, 512)
Image  : (12400, 512)
Labels : (12400,)
Numeric: (12400, 3)
HasImg : (12400, 1)
Domain : (12400, 1)


NUMERIC METADATA BY CLASS

Class 0 | N = 4,123
score            mean=   0.0215 std=   0.9583 median=  -0.1880
num_comments     mean=   0.7532 std=   1.0123 median=   0.6753
upvote_ratio     mean=   0.7292 std=   0.2523 median=   0.7745

Class 1 | N = 701
score            mean=  -0.0629 std=   0.8606 median=  -0.2722
num_comments     mean=  -0.0439 std=   0.7398 median=  -0.1273
upvote_ratio     mean=   0.8815 std=   0.2247 median=   0.9381

Class 2 | N = 2,457
score            mean=   0.1876 std=   1.0290 median=  -0.0188
num_comments     mean=   0.0404 std=   0.9493 median=  -0.1273
upvote_ratio     mean=   0.8562 std=   0.2383 median=   0.9147

Class 3 | N = 140
score            mean=   0.5641 std=   1.3686 median=   0.037

# HGR+++ NEAR-DUPLICATE TEXT SENSITIVITY ANALYSIS

In [13]:
# ============================================================
# cell- 5
# HGR+++ NEAR-DUPLICATE TEXT SENSITIVITY ANALYSIS
# EXACT SAVED CHECKPOINT ARCHITECTURE
#
# NO RETRAINING
#
# Tests the SAME Epoch-6 / Seed-42 model on:
#
#   1. Full test
#   2. Remove CLIP text similarity >= 0.95
#   3. Remove CLIP text similarity >= 0.98
#   4. Remove CLIP text similarity >= 0.99
#   5. Remove CLIP text similarity >= 0.995
#
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. CONFIG
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/fakeddit1-precomputed-features"
)

TEST_CSV = (
    "/kaggle/input/datasets/"
    "mariaislambarra/new-124k-data/"
    "test_clean_filled_12400.csv"
)

SIMILARITY_CSV = (
    "/kaggle/working/"
    "fakeddit_train_test_text_nearest_neighbors.csv"
)

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_leakage_sensitivity_seed42.pt"
)

OUTPUT_CSV = (
    "/kaggle/working/"
    "HGR_plus_plus_near_duplicate_sensitivity.csv"
)

THRESHOLDS = [
    0.95,
    0.98,
    0.99,
    0.995
]

BATCH_SIZE = 256


print("=" * 80)
print("HGR+++ NEAR-DUPLICATE TEXT SENSITIVITY ANALYSIS")
print("=" * 80)

print("Device:", DEVICE)


# ============================================================
# 2. LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("\nCheckpoint:")
print("Epoch             :", checkpoint["epoch"])
print("Seed              :", checkpoint["seed"])
print(
    "Best Val Macro-F1 :",
    checkpoint["best_val_macro_f1"]
)
print("Config            :", checkpoint["config"])


# ============================================================
# 3. LOAD DATA
# ============================================================

def load_tensor(filename):

    return torch.load(
        os.path.join(
            FEAT_DIR,
            filename
        ),
        map_location="cpu"
    )


test_text = load_tensor(
    "test_text.pt"
).float()

test_image = load_tensor(
    "test_image.pt"
).float()

test_y = load_tensor(
    "test_labels.pt"
).long()

test_num = load_tensor(
    "test_num.pt"
).float()

test_has = load_tensor(
    "test_has.pt"
).float()

test_cat = load_tensor(
    "test_cat.pt"
).long()


train_cat = load_tensor(
    "train_cat.pt"
).long()

val_cat = load_tensor(
    "val_cat.pt"
).long()


test_df = pd.read_csv(
    TEST_CSV
)

similarity_df = pd.read_csv(
    SIMILARITY_CSV
)


print("\nDataset:")
print("Test CSV        :", len(test_df))
print("Similarity rows :", len(similarity_df))
print("Test tensor     :", len(test_y))


assert len(test_df) == 12400
assert len(similarity_df) == 12400
assert len(test_y) == 12400


# ============================================================
# 4. VERIFY TEST ORDER
# ============================================================

csv_labels = (
    pd.to_numeric(
        test_df["6_way_label"],
        errors="raise"
    )
    .astype(int)
    .to_numpy()
)

tensor_labels = (
    test_y
    .numpy()
    .astype(int)
)

assert np.array_equal(
    csv_labels,
    tensor_labels
), (
    "STOP: Test CSV and tensor labels "
    "are not aligned."
)


assert np.array_equal(
    similarity_df[
        "test_index"
    ].to_numpy(),
    np.arange(12400)
), (
    "STOP: Similarity results are not "
    "aligned with test rows."
)


print(
    "✓ CSV ↔ tensors ↔ similarity results "
    "are aligned."
)


# ============================================================
# 5. CATEGORY VOCABULARY
# ============================================================

CAT_VOCAB = int(
    max(
        train_cat.max().item(),
        val_cat.max().item(),
        test_cat.max().item()
    )
    + 1
)

print(
    "Domain vocabulary:",
    CAT_VOCAB
)

assert CAT_VOCAB == 1270, (
    f"Expected 1270 domains, got {CAT_VOCAB}"
)




class HGR_Checkpoint_Exact(nn.Module):

    def __init__(
        self,
        num_classes=6,
        cat_vocab=1270,
        cat_emb_dim=32,
        dropout=0.20,
        gate_tau=2.0
    ):

        super().__init__()

        self.gate_tau = gate_tau

        # ----------------------------------------------------
        # Domain embedding
        # checkpoint:
        # cat_embs.0.weight = (1270, 32)
        # ----------------------------------------------------

        self.cat_embs = nn.ModuleList([
            nn.Embedding(
                cat_vocab,
                cat_emb_dim
            )
        ])


        # ----------------------------------------------------
        # CLIP:
        # text 512 + image 512 = 1024
        #
        # checkpoint:
        # clip_compress.0.weight = (128, 1024)
        # ----------------------------------------------------

        self.clip_compress = nn.Sequential(

            nn.Linear(
                1024,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # ----------------------------------------------------
        # Metadata/context:
        # ----------------------------------------------------

        self.meta_mlp = nn.Sequential(

            nn.Linear(
                164,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # ----------------------------------------------------
        # Gate
        # ----------------------------------------------------

        self.gate_logits = nn.Sequential(

            nn.Linear(
                256,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                1280
            )
        )


        # ----------------------------------------------------
        # Classification head
        #
        # checkpoint:
        #
        # head.0 = 1280 -> 1024
        # head.3 = 1024 -> 6
        # ----------------------------------------------------

        self.head = nn.Sequential(

            nn.Linear(
                1280,
                1024
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                1024,
                num_classes
            )
        )


    def forward(
        self,
        text,
        image,
        num,
        has,
        cat
    ):

        # ----------------------------------------------------
        # CLIP normalization
        # ----------------------------------------------------

        text = F.normalize(
            text,
            p=2,
            dim=1
        )

        image = F.normalize(
            image,
            p=2,
            dim=1
        )


        # ----------------------------------------------------
        # Text + image
        # ----------------------------------------------------

        clip = torch.cat(
            [
                text,
                image
            ],
            dim=1
        )


        # ----------------------------------------------------
        # CLIP compression
        # ----------------------------------------------------

        clip_small = self.clip_compress(
            clip
        )


        # ----------------------------------------------------
        # Domain embedding
        # ----------------------------------------------------

        domain_emb = self.cat_embs[0](
            cat[:, 0]
        )


        # ----------------------------------------------------
        # Context representation
        # ----------------------------------------------------

        meta_input = torch.cat(
            [
                num,
                has,
                domain_emb,
                clip_small
            ],
            dim=1
        )

        meta = self.meta_mlp(
            meta_input
        )


        # ----------------------------------------------------
        # Base fused representation
        #
        # 1024 + 256 = 1280
        # ----------------------------------------------------

        z0 = torch.cat(
            [
                clip,
                meta
            ],
            dim=1
        )


        # ----------------------------------------------------
        # Feature-wise gate
        # ----------------------------------------------------

        gate_logits = self.gate_logits(
            meta
        )

        gate = torch.sigmoid(
            gate_logits
            /
            self.gate_tau
        )


        # ----------------------------------------------------
        # Centered residual modulation
        # ----------------------------------------------------

        z = (
            z0
            +
            z0
            *
            (
                gate
                - 0.5
            )
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.head(
            z
        )

        return logits, gate


# ============================================================
# 7. INITIALIZE MODEL
# ============================================================

model_exact = HGR_Checkpoint_Exact(
    num_classes=6,
    cat_vocab=CAT_VOCAB,
    cat_emb_dim=32,
    dropout=checkpoint[
        "config"
    ]["dropout"],
    gate_tau=checkpoint[
        "config"
    ]["gate_tau"]
).to(DEVICE)


# ============================================================
# 8. LOAD EXACT SAVED WEIGHTS
# ============================================================

state_dict = checkpoint[
    "model_state_dict"
]

model_exact.load_state_dict(
    state_dict,
    strict=True
)

model_exact.eval()

print(
    "\n✓ Exact checkpoint loaded with strict=True."
)


# ============================================================
# 9. DATASET
# ============================================================

class TestDataset(Dataset):

    def __init__(
        self,
        text,
        image,
        labels,
        num,
        has,
        cat
    ):

        self.text = text
        self.image = image
        self.labels = labels
        self.num = num
        self.has = has
        self.cat = cat


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.text[idx],
            self.image[idx],
            self.labels[idx],
            self.num[idx],
            self.has[idx],
            self.cat[idx]
        )


test_dataset = TestDataset(
    test_text,
    test_image,
    test_y,
    test_num,
    test_has,
    test_cat
)


# ============================================================
# 10. EVALUATION FUNCTION
# ============================================================

def evaluate(
    indices,
    evaluation_name
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    subset = Subset(
        test_dataset,
        indices.tolist()
    )

    loader = DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    all_true = []
    all_pred = []
    all_prob = []


    model_exact.eval()


    with torch.no_grad():

        for (
            text,
            image,
            labels,
            num,
            has,
            cat
        ) in loader:


            text = text.to(
                DEVICE
            )

            image = image.to(
                DEVICE
            )

            num = num.to(
                DEVICE
            )

            has = has.to(
                DEVICE
            )

            cat = cat.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )


            logits, gate = model_exact(
                text,
                image,
                num,
                has,
                cat
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            predictions = torch.argmax(
                logits,
                dim=1
            )


            all_true.append(
                labels.cpu()
            )

            all_pred.append(
                predictions.cpu()
            )

            all_prob.append(
                probabilities.cpu()
            )


    y_true = torch.cat(
        all_true
    ).numpy()

    y_pred = torch.cat(
        all_pred
    ).numpy()

    y_prob = torch.cat(
        all_prob
    ).numpy()


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    try:

        macro_auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        macro_auc = np.nan


    return {

        "Evaluation":
            evaluation_name,

        "N":
            len(y_true),

        "Excluded":
            12400 - len(y_true),

        "Accuracy":
            accuracy,

        "Macro_Precision":
            precision,

        "Macro_Recall":
            recall,

        "Macro_F1":
            macro_f1,

        "Weighted_F1":
            weighted_f1,

        "Macro_AUC":
            macro_auc,

        "y_true":
            y_true,

        "y_pred":
            y_pred
    }


# ============================================================
# 11. FULL TEST
# ============================================================

all_indices = np.arange(
    12400
)

results = []


full = evaluate(
    all_indices,
    "Full Test"
)

results.append(
    full
)


print("\n")
print("=" * 80)
print("FULL TEST CHECK")
print("=" * 80)

print(
    f"Accuracy    : {full['Accuracy']:.6f}"
)

print(
    f"Macro-F1    : {full['Macro_F1']:.6f}"
)

print(
    f"Weighted-F1 : {full['Weighted_F1']:.6f}"
)

print(
    f"Macro-AUC   : {full['Macro_AUC']:.6f}"
)


# ============================================================
# 12. SANITY CHECK
#
# Earlier checkpoint evaluation:
#
# Accuracy  ~ 0.9638
# Macro-F1  ~ 0.9352
# Macro-AUC ~ 0.9947
#
# Do not continue interpreting results if these differ
# substantially.
# ============================================================

EXPECTED_ACC = 0.9638
EXPECTED_F1 = 0.9352
EXPECTED_AUC = 0.9947


if (
    abs(
        full["Accuracy"]
        -
        EXPECTED_ACC
    ) > 0.002
    or
    abs(
        full["Macro_F1"]
        -
        EXPECTED_F1
    ) > 0.002
    or
    abs(
        full["Macro_AUC"]
        -
        EXPECTED_AUC
    ) > 0.002
):

    raise RuntimeError(
        "\nSTOP: Full-test checkpoint result does not "
        "reproduce the previous evaluation.\n"
        "Do NOT interpret sensitivity results."
    )


print(
    "\n✓ Full-test checkpoint result reproduced."
)


# ============================================================
# 13. GET SIMILARITIES
# ============================================================

similarities = (
    similarity_df[
        "cosine_similarity"
    ]
    .astype(float)
    .to_numpy()
)


# ============================================================
# 14. RUN THRESHOLD SENSITIVITY
# ============================================================

for threshold in THRESHOLDS:

    # Keep only samples BELOW threshold

    keep_indices = np.where(
        similarities
        <
        threshold
    )[0]


    excluded = (
        12400
        -
        len(
            keep_indices
        )
    )


    result = evaluate(
        keep_indices,
        f"Remove >= {threshold}"
    )


    results.append(
        result
    )


    print("\n" + "=" * 80)

    print(
        f"REMOVE TEXT SIMILARITY >= {threshold}"
    )

    print("=" * 80)

    print(
        f"Excluded    : {excluded:,}"
    )

    print(
        f"Remaining   : {len(keep_indices):,}"
    )

    print(
        f"Accuracy    : {result['Accuracy']:.6f}"
    )

    print(
        f"Macro-F1    : {result['Macro_F1']:.6f}"
    )

    print(
        f"Weighted-F1 : {result['Weighted_F1']:.6f}"
    )

    print(
        f"Macro-AUC   : {result['Macro_AUC']:.6f}"
    )


# ============================================================
# 15. SUMMARY
# ============================================================

summary = []


for result in results:

    summary.append({

        "Evaluation":
            result["Evaluation"],

        "N":
            result["N"],

        "Excluded":
            result["Excluded"],

        "Accuracy":
            result["Accuracy"],

        "Accuracy_Delta":
            (
                result["Accuracy"]
                -
                full["Accuracy"]
            ),

        "Macro_Precision":
            result["Macro_Precision"],

        "Macro_Recall":
            result["Macro_Recall"],

        "Macro_F1":
            result["Macro_F1"],

        "Macro_F1_Delta":
            (
                result["Macro_F1"]
                -
                full["Macro_F1"]
            ),

        "Weighted_F1":
            result["Weighted_F1"],

        "Macro_AUC":
            result["Macro_AUC"],

        "Macro_AUC_Delta":
            (
                result["Macro_AUC"]
                -
                full["Macro_AUC"]
            )
    })


summary_df = pd.DataFrame(
    summary
)


# ============================================================
# 16. PRINT FINAL TABLE
# ============================================================

print("\n")
print("=" * 110)

print(
    "FINAL NEAR-DUPLICATE TEXT SENSITIVITY RESULTS"
)

print("=" * 110)


print(
    f"{'Evaluation':<20}"
    f"{'N':>10}"
    f"{'Excluded':>12}"
    f"{'Accuracy':>12}"
    f"{'Δ Acc':>11}"
    f"{'Macro-F1':>12}"
    f"{'Δ F1':>11}"
    f"{'AUC':>12}"
    f"{'Δ AUC':>11}"
)

print("-" * 110)


for _, row in summary_df.iterrows():

    print(
        f"{row['Evaluation']:<20}"
        f"{int(row['N']):>10,}"
        f"{int(row['Excluded']):>12,}"
        f"{row['Accuracy']:>12.4f}"
        f"{row['Accuracy_Delta']:>+11.4f}"
        f"{row['Macro_F1']:>12.4f}"
        f"{row['Macro_F1_Delta']:>+11.4f}"
        f"{row['Macro_AUC']:>12.4f}"
        f"{row['Macro_AUC_Delta']:>+11.4f}"
    )


# ============================================================
# 17. SAVE
# ============================================================

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)


print("\nSaved:")
print(
    OUTPUT_CSV
)


# ============================================================
# 18. AGGRESSIVE >= 0.95 EXCLUSION REPORT
# ============================================================

aggressive = results[1]


print("\n")
print("=" * 80)

print(
    "CLASSIFICATION REPORT — "
    "AFTER REMOVING SIMILARITY >= 0.95"
)

print("=" * 80)


print(
    classification_report(
        aggressive["y_true"],
        aggressive["y_pred"],
        digits=4,
        zero_division=0
    )
)


print(
    "\nCONFUSION MATRIX"
)

print(
    confusion_matrix(
        aggressive["y_true"],
        aggressive["y_pred"]
    )
)

HGR+++ NEAR-DUPLICATE TEXT SENSITIVITY ANALYSIS
Device: cuda

Checkpoint:
Epoch             : 6
Seed              : 42
Best Val Macro-F1 : 0.9439153546708879
Config            : {'epochs': 6, 'batch_train': 128, 'batch_eval': 256, 'lr': 0.0002, 'weight_decay': 0.0001, 'label_smoothing': 0.02, 'dropout': 0.2, 'gate_tau': 2.0, 'gate_lambda': 0.005, 'gate_target': 0.6}

Dataset:
Test CSV        : 12400
Similarity rows : 12400
Test tensor     : 12400
✓ CSV ↔ tensors ↔ similarity results are aligned.
Domain vocabulary: 1270

✓ Exact checkpoint loaded with strict=True.


FULL TEST CHECK
Accuracy    : 0.963790
Macro-F1    : 0.935197
Weighted-F1 : 0.963611
Macro-AUC   : 0.994659

✓ Full-test checkpoint result reproduced.

REMOVE TEXT SIMILARITY >= 0.95
Excluded    : 2,209
Remaining   : 10,191
Accuracy    : 0.958297
Macro-F1    : 0.934247
Weighted-F1 : 0.958071
Macro-AUC   : 0.994264

REMOVE TEXT SIMILARITY >= 0.98
Excluded    : 497
Remaining   : 11,903
Accuracy    : 0.962614
Macro-F1    : 0.93

# Train ↔ Test Near-Duplicate Text Audit

In [9]:
# ============================================================
# cell- 4
# TRAIN ↔ TEST NEAR-DUPLICATE TEXT AUDIT
# Using the SAME precomputed CLIP text embeddings used by HGR+++
#
# Purpose:
#   1. For every test sample, find its most similar training sample
#   2. Measure cosine similarity
#   3. Report similarity thresholds
#   4. Show strongest candidate pairs
#   5. Compare labels
#   6. Save all results + high-similarity candidates

# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ============================================================
# 1. PATHS — EXACT FINAL DATA USED BY HGR+++
# ============================================================

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/fakeddit1-precomputed-features"
)

TRAIN_CSV = (
    "/kaggle/input/datasets/"
    "mariaislambarra/new-124k-data/"
    "train_clean_filled_99200.csv"
)

TEST_CSV = (
    "/kaggle/input/datasets/"
    "mariaislambarra/new-124k-data/"
    "test_clean_filled_12400.csv"
)

ALL_RESULTS_PATH = (
    "/kaggle/working/"
    "fakeddit_train_test_text_nearest_neighbors.csv"
)

HIGH_SIM_PATH = (
    "/kaggle/working/"
    "fakeddit_high_similarity_text_pairs.csv"
)


# ============================================================
# 2. LOAD 
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("=" * 80)
print("DATASET")
print("=" * 80)

print("Train:", len(train_df))
print("Test :", len(test_df))

assert len(train_df) == 99200, (
    f"Unexpected train size: {len(train_df)}"
)

assert len(test_df) == 12400, (
    f"Unexpected test size: {len(test_df)}"
)


# ============================================================
# 3. LOAD SAME CLIP TEXT EMBEDDINGS 
# ============================================================

train_text = torch.load(
    os.path.join(FEAT_DIR, "train_text.pt"),
    map_location="cpu"
)

test_text = torch.load(
    os.path.join(FEAT_DIR, "test_text.pt"),
    map_location="cpu"
)

print("\nCLIP text embedding shapes:")
print("Train:", train_text.shape)
print("Test :", test_text.shape)

assert len(train_text) == len(train_df)
assert len(test_text) == len(test_df)


# ============================================================
# 4. VERIFY CSV ↔ TENSOR LABEL ALIGNMENT
# ============================================================

train_tensor_labels = torch.load(
    os.path.join(FEAT_DIR, "train_labels.pt"),
    map_location="cpu"
).cpu().numpy().astype(int)

test_tensor_labels = torch.load(
    os.path.join(FEAT_DIR, "test_labels.pt"),
    map_location="cpu"
).cpu().numpy().astype(int)

train_csv_labels = (
    pd.to_numeric(
        train_df["6_way_label"],
        errors="raise"
    )
    .astype(int)
    .to_numpy()
)

test_csv_labels = (
    pd.to_numeric(
        test_df["6_way_label"],
        errors="raise"
    )
    .astype(int)
    .to_numpy()
)

assert np.array_equal(
    train_csv_labels,
    train_tensor_labels
), "STOP: Train CSV rows do not match train tensors!"

assert np.array_equal(
    test_csv_labels,
    test_tensor_labels
), "STOP: Test CSV rows do not match test tensors!"

print("\n✓ CSV ↔ tensor label alignment verified.")


# ============================================================
# 5. NORMALIZE CLIP EMBEDDINGS
# ============================================================


train_text = F.normalize(
    train_text.float(),
    p=2,
    dim=1
)

test_text = F.normalize(
    test_text.float(),
    p=2,
    dim=1
)

print("✓ CLIP text embeddings L2-normalized.")


# ============================================================
# 6. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 7. FIND NEAREST TRAINING TEXT FOR EVERY TEST SAMPLE

# ============================================================

BATCH_SIZE = 128

train_gpu = train_text.to(device)

best_similarities = []
best_train_indices = []

print("\n" + "=" * 80)
print("SEARCHING FOR NEAREST TRAINING TEXT")
print("=" * 80)

with torch.no_grad():

    for start in range(
        0,
        len(test_text),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(test_text)
        )

        test_batch = (
            test_text[start:end]
            .to(device)
        )

        # cosine similarity because vectors are normalized
        similarity = (
            test_batch
            @
            train_gpu.T
        )

        max_sim, max_idx = (
            similarity.max(dim=1)
        )

        best_similarities.append(
            max_sim.cpu()
        )

        best_train_indices.append(
            max_idx.cpu()
        )

        if (
            start == 0
            or end == len(test_text)
            or start % 1280 == 0
        ):
            print(
                f"Processed "
                f"{end:,} / {len(test_text):,}"
            )


best_similarities = (
    torch.cat(
        best_similarities
    )
    .numpy()
)

best_train_indices = (
    torch.cat(
        best_train_indices
    )
    .numpy()
)


print("\n✓ Nearest-neighbor search completed.")


# ============================================================
# 8. HELPER FOR SAFE TEXT
# ============================================================

def safe_text(value):

    if pd.isna(value):
        return ""

    return str(value).strip()


# ============================================================
# 9. BUILD TRAIN ↔ TEST PAIR TABLE
# ============================================================

records = []

for test_idx in range(len(test_df)):

    train_idx = int(
        best_train_indices[test_idx]
    )

    similarity = float(
        best_similarities[test_idx]
    )

    test_row = test_df.iloc[test_idx]
    train_row = train_df.iloc[train_idx]

    test_clean = safe_text(
        test_row.get(
            "clean_title",
            ""
        )
    )

    train_clean = safe_text(
        train_row.get(
            "clean_title",
            ""
        )
    )

    test_raw = safe_text(
        test_row.get(
            "title",
            ""
        )
    )

    train_raw = safe_text(
        train_row.get(
            "title",
            ""
        )
    )

    test_label = int(
        test_row["6_way_label"]
    )

    train_label = int(
        train_row["6_way_label"]
    )

    records.append(
        {
            "test_index":
                test_idx,

            "train_index":
                train_idx,

            "cosine_similarity":
                similarity,

            "test_id":
                test_row.get(
                    "id",
                    ""
                ),

            "train_id":
                train_row.get(
                    "id",
                    ""
                ),

            "test_label":
                test_label,

            "train_label":
                train_label,

            "same_label":
                test_label == train_label,

            "test_raw_title":
                test_raw,

            "train_raw_title":
                train_raw,

            "test_clean_title":
                test_clean,

            "train_clean_title":
                train_clean,

            "exact_clean_title":
                (
                    test_clean
                    ==
                    train_clean
                    and
                    test_clean != ""
                )
        }
    )


results = pd.DataFrame(records)


# ============================================================
# 10. BASIC SIMILARITY DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("MAXIMUM TRAIN SIMILARITY FOR EACH TEST SAMPLE")
print("=" * 80)

print(
    results[
        "cosine_similarity"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)


# ============================================================
# 11. THRESHOLD ANALYSIS
# ============================================================

thresholds = [
    0.80,
    0.85,
    0.90,
    0.95,
    0.98,
    0.99,
    0.995
]

print("\n" + "=" * 80)
print("HIGH-SIMILARITY THRESHOLDS")
print("=" * 80)

threshold_records = []

for threshold in thresholds:

    subset = results[
        results["cosine_similarity"]
        >= threshold
    ]

    n = len(subset)

    percentage = (
        100 * n / len(results)
    )

    if n > 0:

        same_label_n = int(
            subset[
                "same_label"
            ].sum()
        )

        same_label_pct = (
            100
            *
            same_label_n
            /
            n
        )

        exact_clean_n = int(
            subset[
                "exact_clean_title"
            ].sum()
        )

    else:

        same_label_n = 0
        same_label_pct = 0
        exact_clean_n = 0

    threshold_records.append(
        {
            "threshold":
                threshold,

            "test_samples":
                n,

            "percentage_of_test":
                percentage,

            "same_label_pairs":
                same_label_n,

            "same_label_percentage":
                same_label_pct,

            "exact_clean_title_pairs":
                exact_clean_n
        }
    )

    print(
        f"Similarity >= {threshold:.3f}: "
        f"{n:,} / {len(results):,} "
        f"({percentage:.4f}%) | "
        f"same-label={same_label_n:,} "
        f"({same_label_pct:.2f}%) | "
        f"exact-clean-title={exact_clean_n:,}"
    )


threshold_df = pd.DataFrame(
    threshold_records
)


# ============================================================
# 12. CHECK EXACT clean_title PAIRS

# ============================================================

exact_clean = results[
    results["exact_clean_title"]
]

print("\n" + "=" * 80)
print("EXACT clean_title AMONG NEAREST NEIGHBORS")
print("=" * 80)

print(
    "Exact clean-title nearest-neighbor pairs:",
    len(exact_clean)
)

print(
    "NOTE: This number does not necessarily have to equal 78 "
    "if another training title has a higher CLIP similarity "
    "than the exact-string match."
)


# ============================================================
# 13. TOP 100 MOST SIMILAR TRAIN ↔ TEST PAIRS
# ============================================================

top100 = (
    results
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .head(100)
    .copy()
)

print("\n" + "=" * 80)
print("TOP 50 MOST SIMILAR TRAIN ↔ TEST PAIRS")
print("=" * 80)

pd.set_option(
    "display.max_colwidth",
    100
)

print(
    top100[
        [
            "cosine_similarity",
            "train_label",
            "test_label",
            "same_label",
            "exact_clean_title",
            "train_clean_title",
            "test_clean_title"
        ]
    ]
    .head(50)
    .to_string(
        index=False
    )
)


# ============================================================
# 14. POTENTIAL NEAR-DUPLICATES
#
# We use >=0.95 as a REVIEW threshold.

# ============================================================

REVIEW_THRESHOLD = 0.95

high_similarity = (
    results[
        results[
            "cosine_similarity"
        ]
        >= REVIEW_THRESHOLD
    ]
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .copy()
)

print("\n" + "=" * 80)
print(
    f"CANDIDATES FOR MANUAL REVIEW "
    f"(similarity >= {REVIEW_THRESHOLD})"
)
print("=" * 80)

print(
    "Candidate test samples:",
    len(high_similarity)
)

print(
    "Percentage of test:",
    f"{100 * len(high_similarity) / len(test_df):.4f}%"
)

if len(high_similarity) > 0:

    print(
        "Same-label candidates:",
        int(
            high_similarity[
                "same_label"
            ].sum()
        ),
        "/",
        len(high_similarity)
    )


# ============================================================
# 15. HIGH-SIMILARITY BUT NOT EXACT clean_title
# ============================================================

near_duplicate_candidates = (
    high_similarity[
        ~high_similarity[
            "exact_clean_title"
        ]
    ]
    .copy()
)

print("\n" + "=" * 80)
print("HIGH-SIMILARITY NON-EXACT-TITLE CANDIDATES")
print("=" * 80)

print(
    "Count:",
    len(
        near_duplicate_candidates
    )
)

if len(
    near_duplicate_candidates
) > 0:

    print(
        near_duplicate_candidates[
            [
                "cosine_similarity",
                "train_label",
                "test_label",
                "same_label",
                "train_clean_title",
                "test_clean_title"
            ]
        ]
        .head(50)
        .to_string(
            index=False
        )
    )


# ============================================================
# 16. SAVE RESULTS
# ============================================================

results.to_csv(
    ALL_RESULTS_PATH,
    index=False
)

high_similarity.to_csv(
    HIGH_SIM_PATH,
    index=False
)

threshold_df.to_csv(
    "/kaggle/working/"
    "fakeddit_text_similarity_threshold_summary.csv",
    index=False
)

near_duplicate_candidates.to_csv(
    "/kaggle/working/"
    "fakeddit_near_duplicate_text_candidates.csv",
    index=False
)


print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print("\nSaved files:")

print(
    "1.",
    ALL_RESULTS_PATH
)

print(
    "2.",
    HIGH_SIM_PATH
)

print(
    "3. /kaggle/working/"
    "fakeddit_text_similarity_threshold_summary.csv"
)

print(
    "4. /kaggle/working/"
    "fakeddit_near_duplicate_text_candidates.csv"
)

print(
    "\nIMPORTANT:"
)

print(
    "Do NOT call every high-similarity pair a duplicate."
)

print(
    "The highest-similarity pairs must be inspected "
    "before making that conclusion."
)

DATASET
Train: 99200
Test : 12400

CLIP text embedding shapes:
Train: torch.Size([99200, 512])
Test : torch.Size([12400, 512])

✓ CSV ↔ tensor label alignment verified.
✓ CLIP text embeddings L2-normalized.
Device: cuda

SEARCHING FOR NEAREST TRAINING TEXT
Processed 128 / 12,400
Processed 1,408 / 12,400
Processed 2,688 / 12,400
Processed 3,968 / 12,400
Processed 5,248 / 12,400
Processed 6,528 / 12,400
Processed 7,808 / 12,400
Processed 9,088 / 12,400
Processed 10,368 / 12,400
Processed 11,648 / 12,400
Processed 12,400 / 12,400

✓ Nearest-neighbor search completed.

MAXIMUM TRAIN SIMILARITY FOR EACH TEST SAMPLE
count    12400.000000
mean         0.874111
std          0.081869
min          0.394531
50%          0.889284
75%          0.936810
90%          0.965118
95%          0.977069
99%          0.990793
99.5%        0.993771
99.9%        0.996929
max          0.999602
Name: cosine_similarity, dtype: float64

HIGH-SIMILARITY THRESHOLDS
Similarity >= 0.800: 10,280 / 12,400 (82.9032%) | 

# HGR+++ LEAKAGE SENSITIVITY EXPERIMENT

In [8]:
# ============================================================
# cell- 3
# HGR+++ LEAKAGE SENSITIVITY EXPERIMENT
# ============================================================

import os
import random
import copy
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Seed  :", SEED)


# ============================================================
# 2. HGR+++ CONFIGURATION
# ============================================================

NUM_LABELS = 6
EPOCHS = 6

BATCH_TRAIN = 128
BATCH_EVAL = 256

GRAD_CLIP = 1.0


LR = 2e-4

WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.02

META_HIDDEN = 256
EMB_DIM = 32
GATE_HIDDEN = 256
HEAD_HIDDEN = 1024

DROPOUT = 0.20
GATE_TAU = 2.0

GATE_LAMBDA = 0.005
GATE_TARGET = 0.6

NUM_COLS = [
    "score",
    "num_comments",
    "upvote_ratio"
]

CAT_COLS = ["domain"]


# ============================================================
# 3. PATHS
# ============================================================

FEAT_DIR = (
    "/kaggle/input/datasets/"
    "mariaislambarra/fakeddit1-precomputed-features"
)

TRAIN_CSV = (
    "/kaggle/input/datasets/"
    "mariaislambarra/new-124k-data/"
    "train_clean_filled_99200.csv"
)

TEST_CSV = (
    "/kaggle/input/datasets/"
    "mariaislambarra/new-124k-data/"
    "test_clean_filled_12400.csv"
)

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_leakage_sensitivity_seed42.pt"
)

RESULT_PATH = (
    "/kaggle/working/"
    "HGR_plus_plus_leakage_sensitivity_results.csv"
)


# ============================================================
# 4. LOAD PRECOMPUTED FEATURES
# ============================================================

def load_split(name):

    t = torch.load(
        os.path.join(FEAT_DIR, f"{name}_text.pt"),
        map_location="cpu"
    )

    v = torch.load(
        os.path.join(FEAT_DIR, f"{name}_image.pt"),
        map_location="cpu"
    )

    y = torch.load(
        os.path.join(FEAT_DIR, f"{name}_labels.pt"),
        map_location="cpu"
    )

    num = torch.load(
        os.path.join(FEAT_DIR, f"{name}_num.pt"),
        map_location="cpu"
    )

    has = torch.load(
        os.path.join(FEAT_DIR, f"{name}_has.pt"),
        map_location="cpu"
    )

    cat = torch.load(
        os.path.join(FEAT_DIR, f"{name}_cat.pt"),
        map_location="cpu"
    )

    return t, v, y, num, has, cat


train_t, train_v, train_y, train_num, train_has, train_cat = (
    load_split("train")
)

val_t, val_v, val_y, val_num, val_has, val_cat = (
    load_split("val")
)

test_t, test_v, test_y, test_num, test_has, test_cat = (
    load_split("test")
)


print("\nFeature shapes:")
print("Train:", train_t.shape, train_y.shape)
print("Val  :", val_t.shape, val_y.shape)
print("Test :", test_t.shape, test_y.shape)


# ============================================================
# 5. SAFETY CHECK:

# ============================================================

train_csv = pd.read_csv(TRAIN_CSV)
test_csv = pd.read_csv(TEST_CSV)

assert len(train_csv) == len(train_y), (
    "Train CSV and train feature tensor sizes do not match!"
)

assert len(test_csv) == len(test_y), (
    "Test CSV and test feature tensor sizes do not match!"
)

csv_test_labels = (
    pd.to_numeric(
        test_csv["6_way_label"],
        errors="raise"
    )
    .astype(int)
    .to_numpy()
)

tensor_test_labels = test_y.cpu().numpy().astype(int)

if not np.array_equal(
    csv_test_labels,
    tensor_test_labels
):
    raise RuntimeError(
        "STOP: Test CSV row order does not match "
        "the precomputed feature tensors."
    )

print(
    "\n✓ CSV/test-feature row-order verification passed."
)


# ============================================================
# 6. IDENTIFY TEST clean_title OVERLAP WITH TRAIN
# ============================================================

def clean_title(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


train_titles = set(
    train_csv["clean_title"]
    .map(clean_title)
)

train_titles.discard("")

test_clean_titles = (
    test_csv["clean_title"]
    .map(clean_title)
)

overlap_mask = (
    test_clean_titles.isin(train_titles)
    &
    (test_clean_titles != "")
)

overlap_indices = np.where(
    overlap_mask.to_numpy()
)[0]

clean_indices = np.where(
    ~overlap_mask.to_numpy()
)[0]


print("\n" + "=" * 70)
print("TEST TITLE OVERLAP")
print("=" * 70)

print(
    "Full test samples       :",
    len(test_csv)
)

print(
    "Overlapping samples     :",
    len(overlap_indices)
)

print(
    "Non-overlapping samples :",
    len(clean_indices)
)

print(
    "Removed percentage      :",
    f"{100 * len(overlap_indices) / len(test_csv):.4f}%"
)

# We expect 78 based on the previous audit.
if len(overlap_indices) != 78:
    print(
        "\nWARNING: Expected 78 overlapping test samples "
        "from the previous audit, but found",
        len(overlap_indices)
    )
else:
    print(
        "✓ Reproduced the expected 78 overlapping test samples."
    )


# ============================================================
# 7. CATEGORY VOCABULARY SIZE
# ============================================================

cat_sizes = []

if train_cat.ndim == 2:
    for i in range(train_cat.size(1)):
        cat_sizes.append(
            int(train_cat[:, i].max().item()) + 1
        )

if len(CAT_COLS) == 0:
    cat_sizes = []

print("\nCategory vocabulary sizes:", cat_sizes)


# ============================================================
# 8. DATASET
# ============================================================

class HGRFeatDataset(Dataset):

    def __init__(
        self,
        t,
        v,
        y,
        num,
        has,
        cat
    ):

        self.t = t
        self.v = v
        self.y = y
        self.num = num
        self.has = has
        self.cat = cat

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):

        return {
            "t": self.t[i],
            "v": self.v[i],
            "labels": self.y[i],
            "num": self.num[i],
            "has_img": self.has[i],
            "cat_ids": self.cat[i]
        }


def collate(batch):

    return {

        "t":
        torch.stack(
            [b["t"] for b in batch]
        ),

        "v":
        torch.stack(
            [b["v"] for b in batch]
        ),

        "labels":
        torch.tensor(
            [int(b["labels"]) for b in batch],
            dtype=torch.long
        ),

        "num":
        torch.stack(
            [b["num"] for b in batch]
        ),

        "has_img":
        torch.stack(
            [b["has_img"] for b in batch]
        ),

        "cat_ids":
        (
            torch.stack(
                [b["cat_ids"] for b in batch]
            )
            if len(CAT_COLS) > 0
            else
            torch.zeros(
                (len(batch), 0),
                dtype=torch.long
            )
        )
    }


train_dataset = HGRFeatDataset(
    train_t,
    train_v,
    train_y,
    train_num,
    train_has,
    train_cat
)

val_dataset = HGRFeatDataset(
    val_t,
    val_v,
    val_y,
    val_num,
    val_has,
    val_cat
)

test_dataset = HGRFeatDataset(
    test_t,
    test_v,
    test_y,
    test_num,
    test_has,
    test_cat
)


# ============================================================
# 9. DATALOADERS
# ============================================================

# Seeded generator for reproducible shuffle
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_TRAIN,
    shuffle=True,
    collate_fn=collate,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_EVAL,
    shuffle=False,
    collate_fn=collate
)

full_test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_EVAL,
    shuffle=False,
    collate_fn=collate
)

clean_test_dataset = Subset(
    test_dataset,
    clean_indices.tolist()
)

clean_test_loader = DataLoader(
    clean_test_dataset,
    batch_size=BATCH_EVAL,
    shuffle=False,
    collate_fn=collate
)


# ============================================================
# 10. ORIGINAL HGR+++ ARCHITECTURE
# ============================================================

class HGR10_HGR_Improved_Feats(nn.Module):

    def __init__(
        self,
        num_labels=6,
        meta_hidden=256,
        emb_dim=32,
        gate_hidden=256,
        head_hidden=1024,
        dropout=0.20,
        gate_tau=2.0
    ):

        super().__init__()

        self.gate_tau = gate_tau

        d = train_t.size(1)

        clip_dim = 2 * d

        self.cat_embs = nn.ModuleList([
            nn.Embedding(
                cat_sizes[i],
                emb_dim
            )
            for i in range(len(CAT_COLS))
        ])

        self.clip_compress = nn.Sequential(
            nn.Linear(
                clip_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        meta_in = (
            len(NUM_COLS)
            + 1
            + len(CAT_COLS) * emb_dim
            + 128
        )

        self.meta_mlp = nn.Sequential(
            nn.Linear(
                meta_in,
                meta_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fused_dim = (
            clip_dim
            + meta_hidden
        )

        self.gate_logits = nn.Sequential(
            nn.Linear(
                meta_hidden,
                gate_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                gate_hidden,
                self.fused_dim
            )
        )

        self.head = nn.Sequential(
            nn.Linear(
                self.fused_dim,
                head_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                head_hidden,
                num_labels
            )
        )

    def forward(
        self,
        t,
        v,
        num,
        has_img,
        cat_ids,
        return_gate=False
    ):

        t = F.normalize(
            t,
            p=2,
            dim=1
        )

        v = F.normalize(
            v,
            p=2,
            dim=1
        )

        z_clip = torch.cat(
            [t, v],
            dim=1
        )

        z_clip_comp = (
            self.clip_compress(
                z_clip
            )
        )

        if len(CAT_COLS):

            cat = torch.cat(
                [
                    emb(
                        cat_ids[:, i]
                    )
                    for i, emb
                    in enumerate(
                        self.cat_embs
                    )
                ],
                dim=1
            )

        else:

            cat = torch.empty(
                (num.size(0), 0),
                device=num.device
            )

        meta_in = torch.cat(
            [
                num,
                has_img,
                cat,
                z_clip_comp
            ],
            dim=1
        )

        meta = self.meta_mlp(
            meta_in
        )

        z0 = torch.cat(
            [
                z_clip,
                meta
            ],
            dim=1
        )

        gl = self.gate_logits(
            meta
        )

        gate = torch.sigmoid(
            gl / self.gate_tau
        )

        z = (
            z0
            +
            z0 * (gate - 0.5)
        )

        logits = self.head(z)

        if return_gate:
            return logits, gate

        return logits


model = HGR10_HGR_Improved_Feats(
    num_labels=NUM_LABELS,
    meta_hidden=META_HIDDEN,
    emb_dim=EMB_DIM,
    gate_hidden=GATE_HIDDEN,
    head_hidden=HEAD_HIDDEN,
    dropout=DROPOUT,
    gate_tau=GATE_TAU
).to(DEVICE)


# ============================================================
# 11. ORIGINAL CLASS-WEIGHTED LABEL-SMOOTHING LOSS
# ============================================================

class LabelSmoothingCE(nn.Module):

    def __init__(
        self,
        smoothing=LABEL_SMOOTH,
        weight=None
    ):

        super().__init__()

        self.smoothing = smoothing
        self.weight = weight

    def forward(
        self,
        logits,
        target
    ):

        log_probs = F.log_softmax(
            logits,
            dim=-1
        )

        n_classes = logits.size(-1)

        with torch.no_grad():

            true_dist = torch.zeros_like(
                log_probs
            )

            true_dist.fill_(
                self.smoothing
                /
                (n_classes - 1)
            )

            true_dist.scatter_(
                1,
                target.unsqueeze(1),
                1.0 - self.smoothing
            )

            if self.weight is not None:

                true_dist = (
                    true_dist
                    *
                    self.weight.view(
                        1,
                        -1
                    )
                )

                true_dist = (
                    true_dist
                    /
                    true_dist.sum(
                        dim=1,
                        keepdim=True
                    )
                )

        return -(
            true_dist
            *
            log_probs
        ).sum(
            dim=1
        ).mean()


counts = torch.bincount(
    train_y,
    minlength=NUM_LABELS
).cpu().numpy()

weights = (
    counts.sum()
    /
    (
        NUM_LABELS
        *
        np.maximum(
            counts,
            1
        )
    )
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float,
    device=DEVICE
)

criterion = LabelSmoothingCE(
    smoothing=LABEL_SMOOTH,
    weight=class_weights
)

print(
    "\nClass counts :",
    counts
)

print(
    "Class weights:",
    weights
)


# ============================================================
# 12. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 13. EVALUATION FUNCTION
# ============================================================

def evaluate(model, loader):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for batch in loader:

            logits = model(
                batch["t"].to(DEVICE),
                batch["v"].to(DEVICE),
                batch["num"].to(DEVICE),
                batch["has_img"].to(DEVICE),
                batch["cat_ids"].to(DEVICE)
            )

            probs = torch.softmax(
                logits,
                dim=1
            )

            preds = logits.argmax(
                dim=1
            )

            y_true.append(
                batch["labels"].cpu()
            )

            y_pred.append(
                preds.cpu()
            )

            y_prob.append(
                probs.cpu()
            )

    y_true = torch.cat(
        y_true
    ).numpy()

    y_pred = torch.cat(
        y_pred
    ).numpy()

    y_prob = torch.cat(
        y_prob
    ).numpy()

    metrics = {}

    metrics["N"] = len(y_true)

    metrics["Accuracy"] = accuracy_score(
        y_true,
        y_pred
    )

    metrics["Macro_Precision"] = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    metrics["Macro_Recall"] = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    metrics["Macro_F1"] = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    metrics["Weighted_F1"] = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # Multiclass macro one-vs-rest AUC
    try:

        metrics["Macro_AUC"] = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro",
            labels=np.arange(NUM_LABELS)
        )

    except Exception as e:

        print(
            "AUC calculation warning:",
            e
        )

        metrics["Macro_AUC"] = np.nan

    return (
        metrics,
        y_true,
        y_pred,
        y_prob
    )


# ============================================================
# 14. TRAIN FOR EXACTLY SIX EPOCHS
#     BEST CHECKPOINT = VALIDATION MACRO-F1
# ============================================================

best_val_macro = -1.0
best_epoch = None

print("\n" + "=" * 70)
print("TRAINING HGR+++")
print("=" * 70)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0
    gate_means = []

    for batch in train_loader:

        logits, gate = model(
            batch["t"].to(DEVICE),
            batch["v"].to(DEVICE),
            batch["num"].to(DEVICE),
            batch["has_img"].to(DEVICE),
            batch["cat_ids"].to(DEVICE),
            return_gate=True
        )

        labels = batch[
            "labels"
        ].to(DEVICE)

        loss = criterion(
            logits,
            labels
        )

        gmean = gate.mean()

        loss = (
            loss
            +
            GATE_LAMBDA
            *
            (
                gmean
                -
                GATE_TARGET
            ).abs()
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        optimizer.step()

        total_loss += (
            loss.item()
        )

        gate_means.append(
            gmean.detach().item()
        )

    val_metrics, _, _, _ = evaluate(
        model,
        val_loader
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={total_loss / len(train_loader):.4f} | "
        f"val_acc={val_metrics['Accuracy']:.4f} | "
        f"val_macroF1={val_metrics['Macro_F1']:.4f} | "
        f"val_weightedF1={val_metrics['Weighted_F1']:.4f} | "
        f"gate_mean={np.mean(gate_means):.3f}"
    )

    # --------------------------------------------------------
    # SAVE BEST CHECKPOINT
    # --------------------------------------------------------

    if (
        val_metrics["Macro_F1"]
        >
        best_val_macro
    ):

        best_val_macro = (
            val_metrics["Macro_F1"]
        )

        best_epoch = (
            epoch + 1
        )

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "epoch":
                    best_epoch,

                "seed":
                    SEED,

                "best_val_macro_f1":
                    best_val_macro,

                "config": {
                    "epochs": EPOCHS,
                    "batch_train": BATCH_TRAIN,
                    "batch_eval": BATCH_EVAL,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "label_smoothing": LABEL_SMOOTH,
                    "dropout": DROPOUT,
                    "gate_tau": GATE_TAU,
                    "gate_lambda": GATE_LAMBDA,
                    "gate_target": GATE_TARGET
                }
            },
            CHECKPOINT_PATH
        )

        print(
            f"  ✓ Saved new best checkpoint "
            f"(epoch {best_epoch}, "
            f"val Macro-F1={best_val_macro:.4f})"
        )


# ============================================================
# 15. RELOAD EXACT BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print("\n" + "=" * 70)
print("BEST CHECKPOINT")
print("=" * 70)

print(
    "Best epoch        :",
    checkpoint["epoch"]
)

print(
    "Best Val Macro-F1 :",
    f"{checkpoint['best_val_macro_f1']:.4f}"
)

print(
    "Checkpoint saved  :",
    CHECKPOINT_PATH
)


# ============================================================
# 16. FULL TEST EVALUATION
# ============================================================

full_metrics, full_true, full_pred, full_prob = evaluate(
    model,
    full_test_loader
)


# ============================================================
# 17. NON-OVERLAPPING TEST EVALUATION
# ============================================================

clean_metrics, clean_true, clean_pred, clean_prob = evaluate(
    model,
    clean_test_loader
)


# ============================================================
# 18. RESULTS
# ============================================================

print("\n" + "=" * 78)
print("FULL TEST vs NON-OVERLAPPING TEST")
print("=" * 78)

print(
    f"{'Metric':<20}"
    f"{'Full Test':>15}"
    f"{'No-Overlap':>15}"
    f"{'Difference':>15}"
)

print("-" * 65)

metric_names = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Weighted_F1",
    "Macro_AUC"
]

for metric in metric_names:

    full_value = full_metrics[
        metric
    ]

    clean_value = clean_metrics[
        metric
    ]

    difference = (
        clean_value
        -
        full_value
    )

    print(
        f"{metric:<20}"
        f"{full_value:>15.4f}"
        f"{clean_value:>15.4f}"
        f"{difference:>+15.4f}"
    )


print("\nSample counts:")

print(
    "Full test       :",
    full_metrics["N"]
)

print(
    "Excluded       :",
    len(overlap_indices)
)

print(
    "Non-overlapping:",
    clean_metrics["N"]
)


# ============================================================
# 19. FULL TEST CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 78)
print("FULL TEST CLASSIFICATION REPORT")
print("=" * 78)

print(
    classification_report(
        full_true,
        full_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 20. NON-OVERLAPPING CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 78)
print("NON-OVERLAPPING TEST CLASSIFICATION REPORT")
print("=" * 78)

print(
    classification_report(
        clean_true,
        clean_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 21. CONFUSION MATRICES
# ============================================================

print("\nFULL TEST CONFUSION MATRIX")
print(
    confusion_matrix(
        full_true,
        full_pred
    )
)

print("\nNON-OVERLAPPING TEST CONFUSION MATRIX")
print(
    confusion_matrix(
        clean_true,
        clean_pred
    )
)


# ============================================================
# 22. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(
    [
        {
            "Evaluation":
                "Full test",

            **full_metrics
        },

        {
            "Evaluation":
                "No train clean_title overlap",

            **clean_metrics
        }
    ]
)

results_df.to_csv(
    RESULT_PATH,
    index=False
)

print("\n" + "=" * 78)
print("EXPERIMENT COMPLETE")
print("=" * 78)

print(
    "\nResults saved to:",
    RESULT_PATH
)

print(
    "Best model saved to:",
    CHECKPOINT_PATH
)

Device: cuda
Seed  : 42

Feature shapes:
Train: torch.Size([99200, 512]) torch.Size([99200])
Val  : torch.Size([12400, 512]) torch.Size([12400])
Test : torch.Size([12400, 512]) torch.Size([12400])

✓ CSV/test-feature row-order verification passed.

TEST TITLE OVERLAP
Full test samples       : 12400
Overlapping samples     : 78
Non-overlapping samples : 12322
Removed percentage      : 0.6290%
✓ Reproduced the expected 78 overlapping test samples.

Category vocabulary sizes: [1270]

Class counts : [32106  5323 19203  1087 39222  2259]
Class weights: [ 0.51496086  3.10601791  0.86097658 15.21005826  0.42153213  7.31887266]

TRAINING HGR+++
Epoch 1/6 | loss=0.8057 | val_acc=0.9494 | val_macroF1=0.8622 | val_weightedF1=0.9479 | gate_mean=0.705
  ✓ Saved new best checkpoint (epoch 1, val Macro-F1=0.8622)
Epoch 2/6 | loss=0.7164 | val_acc=0.9575 | val_macroF1=0.9101 | val_weightedF1=0.9571 | gate_mean=0.608
  ✓ Saved new best checkpoint (epoch 2, val Macro-F1=0.9101)
Epoch 3/6 | loss=0.7014 |

In [6]:
# ============================================================
#cell- 2
# VERIFY OLD DATASET vs CURRENT DATAFRAMES
# ============================================================

import pandas as pd

TRAIN_CSV = "/kaggle/input/datasets/mariaislambarra/new-124k-data/train_clean_filled_99200.csv"
VAL_CSV   = "/kaggle/input/datasets/mariaislambarra/new-124k-data/val_clean_filled_12400.csv"
TEST_CSV  = "/kaggle/input/datasets/mariaislambarra/new-124k-data/test_clean_filled_12400.csv"

tr = pd.read_csv(TRAIN_CSV)
va = pd.read_csv(VAL_CSV)
te = pd.read_csv(TEST_CSV)

print("="*70)
print("FILES")
print("="*70)
print(TRAIN_CSV)
print(VAL_CSV)
print(TEST_CSV)

print("\nSizes:", len(tr), len(va), len(te))

def clean_set(df, col):
    return set(
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    ) - {""}

def normalized_set(df, col):
    return set(
        df[col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    ) - {""}

print("\n" + "="*70)
print("EXACT RAW TITLE OVERLAP")
print("="*70)
print("Train-Val :", len(clean_set(tr, "title") & clean_set(va, "title")))
print("Train-Test:", len(clean_set(tr, "title") & clean_set(te, "title")))
print("Val-Test  :", len(clean_set(va, "title") & clean_set(te, "title")))

print("\n" + "="*70)
print("EXACT clean_title OVERLAP")
print("="*70)
print("Train-Val :", len(clean_set(tr, "clean_title") & clean_set(va, "clean_title")))
print("Train-Test:", len(clean_set(tr, "clean_title") & clean_set(te, "clean_title")))
print("Val-Test  :", len(clean_set(va, "clean_title") & clean_set(te, "clean_title")))

print("\n" + "="*70)
print("NORMALIZED clean_title OVERLAP")
print("="*70)
print("Train-Val :", len(normalized_set(tr, "clean_title") & normalized_set(va, "clean_title")))
print("Train-Test:", len(normalized_set(tr, "clean_title") & normalized_set(te, "clean_title")))
print("Val-Test  :", len(normalized_set(va, "clean_title") & normalized_set(te, "clean_title")))

print("\n" + "="*70)
print("EXACT IMAGE URL OVERLAP")
print("="*70)
print("Train-Val :", len(clean_set(tr, "image_url") & clean_set(va, "image_url")))
print("Train-Test:", len(clean_set(tr, "image_url") & clean_set(te, "image_url")))
print("Val-Test  :", len(clean_set(va, "image_url") & clean_set(te, "image_url")))

print("\n" + "="*70)
print("LABEL COUNTS")
print("="*70)

for name, df in [("Train", tr), ("Val", va), ("Test", te)]:
    print(f"\n{name}")
    print(df["6_way_label"].value_counts().sort_index())

FILES
/kaggle/input/datasets/mariaislambarra/new-124k-data/train_clean_filled_99200.csv
/kaggle/input/datasets/mariaislambarra/new-124k-data/val_clean_filled_12400.csv
/kaggle/input/datasets/mariaislambarra/new-124k-data/test_clean_filled_12400.csv

Sizes: 99200 12400 12400

EXACT RAW TITLE OVERLAP
Train-Val : 0
Train-Test: 0
Val-Test  : 0

EXACT clean_title OVERLAP
Train-Val : 84
Train-Test: 75
Val-Test  : 6

NORMALIZED clean_title OVERLAP
Train-Val : 84
Train-Test: 75
Val-Test  : 6

EXACT IMAGE URL OVERLAP
Train-Val : 0
Train-Test: 0
Val-Test  : 0

LABEL COUNTS

Train
6_way_label
0    32106
1     5323
2    19203
3     1087
4    39222
5     2259
Name: count, dtype: int64

Val
6_way_label
0    4062
1     694
2    2434
3     134
4    4818
5     258
Name: count, dtype: int64

Test
6_way_label
0    4123
1     701
2    2457
3     140
4    4682
5     297
Name: count, dtype: int64


# FINAL FAKEDDIT clean_title OVERLAP INVESTIGATION

In [7]:
# ============================================================
# cell -1
# FINAL FAKEDDIT clean_title OVERLAP INVESTIGATION
# Uses ONLY the exact final CSVs used for HGR+++ experiments
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD EXACT FINAL DATASETS
# ------------------------------------------------------------

TRAIN_CSV = "/kaggle/input/datasets/mariaislambarra/new-124k-data/train_clean_filled_99200.csv"
VAL_CSV   = "/kaggle/input/datasets/mariaislambarra/new-124k-data/val_clean_filled_12400.csv"
TEST_CSV  = "/kaggle/input/datasets/mariaislambarra/new-124k-data/test_clean_filled_12400.csv"

train_final = pd.read_csv(TRAIN_CSV)
val_final   = pd.read_csv(VAL_CSV)
test_final  = pd.read_csv(TEST_CSV)

TITLE_COL = "clean_title"
RAW_TITLE_COL = "title"
LABEL_COL = "6_way_label"
ID_COL = "id"
URL_COL = "image_url"


# ------------------------------------------------------------
# 2. CLEAN TEXT SAFELY
# ------------------------------------------------------------

def clean_text_series(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

for df in [train_final, val_final, test_final]:
    df["_clean_title_check"] = clean_text_series(df[TITLE_COL])


# ------------------------------------------------------------
# 3. ANALYZE clean_title OVERLAP
# ------------------------------------------------------------

def analyze_overlap(train_df, eval_df, eval_name):

    train_titles = set(
        train_df.loc[
            train_df["_clean_title_check"] != "",
            "_clean_title_check"
        ]
    )

    eval_titles = set(
        eval_df.loc[
            eval_df["_clean_title_check"] != "",
            "_clean_title_check"
        ]
    )

    overlap = train_titles & eval_titles

    affected = eval_df[
        eval_df["_clean_title_check"].isin(overlap)
    ].copy()

    print("\n" + "=" * 80)
    print(f"Train ↔ {eval_name} clean_title OVERLAP")
    print("=" * 80)

    print(f"Unique overlapping clean_titles : {len(overlap):,}")
    print(
        f"{eval_name} samples affected          : "
        f"{len(affected):,} / {len(eval_df):,} "
        f"({100 * len(affected) / len(eval_df):.4f}%)"
    )

    # --------------------------------------------------------
    # TITLE-LEVEL LABEL CONSISTENCY
    # --------------------------------------------------------

    rows = []

    for title in sorted(overlap):

        tr_rows = train_df[
            train_df["_clean_title_check"] == title
        ]

        ev_rows = eval_df[
            eval_df["_clean_title_check"] == title
        ]

        train_labels = sorted(
            tr_rows[LABEL_COL].dropna().unique().tolist()
        )

        eval_labels = sorted(
            ev_rows[LABEL_COL].dropna().unique().tolist()
        )

        exact_single_label_match = (
            len(train_labels) == 1
            and len(eval_labels) == 1
            and train_labels == eval_labels
        )

        rows.append({
            "clean_title": title,
            "title_length": len(title),
            "train_count": len(tr_rows),
            f"{eval_name.lower()}_count": len(ev_rows),
            "train_labels": str(train_labels),
            f"{eval_name.lower()}_labels": str(eval_labels),
            "exact_single_label_match": exact_single_label_match
        })

    details = pd.DataFrame(rows)

    if len(details) > 0:

        same = details["exact_single_label_match"].sum()

        print(
            f"Exact single-label matches       : "
            f"{same:,} / {len(details):,} "
            f"({100 * same / len(details):.2f}%)"
        )

        # ----------------------------------------------------
        # TITLE LENGTH
        # ----------------------------------------------------

        print("\nTITLE LENGTH DISTRIBUTION")

        for threshold in [5, 10, 15, 20, 30, 50]:

            n = (details["title_length"] <= threshold).sum()

            print(
                f"Titles <= {threshold:2d} characters : "
                f"{n:,} / {len(details):,} "
                f"({100 * n / len(details):.2f}%)"
            )

        # ----------------------------------------------------
        # MOST FREQUENT OVERLAPS
        # ----------------------------------------------------

        details["total_count"] = (
            details["train_count"]
            + details[f"{eval_name.lower()}_count"]
        )

        print("\nMOST FREQUENT OVERLAPPING clean_titles")
        print("-" * 80)

        cols = [
            "clean_title",
            "title_length",
            "train_count",
            f"{eval_name.lower()}_count",
            "train_labels",
            f"{eval_name.lower()}_labels",
            "exact_single_label_match"
        ]

        print(
            details.sort_values(
                "total_count",
                ascending=False
            )[cols]
            .head(30)
            .to_string(index=False)
        )

    return details, affected


# ------------------------------------------------------------
# 4. TRAIN ↔ VALIDATION
# ------------------------------------------------------------

val_details, val_affected = analyze_overlap(
    train_final,
    val_final,
    "Val"
)


# ------------------------------------------------------------
# 5. TRAIN ↔ TEST  <-- MOST IMPORTANT
# ------------------------------------------------------------

test_details, test_affected = analyze_overlap(
    train_final,
    test_final,
    "Test"
)


# ------------------------------------------------------------
# 6. VAL ↔ TEST
# ------------------------------------------------------------

val_titles = set(
    val_final.loc[
        val_final["_clean_title_check"] != "",
        "_clean_title_check"
    ]
)

test_titles = set(
    test_final.loc[
        test_final["_clean_title_check"] != "",
        "_clean_title_check"
    ]
)

vt_overlap = val_titles & test_titles

print("\n" + "=" * 80)
print("Val ↔ Test clean_title OVERLAP")
print("=" * 80)

print(
    f"Unique overlapping clean_titles : "
    f"{len(vt_overlap):,}"
)

vt_affected = test_final[
    test_final["_clean_title_check"].isin(vt_overlap)
]

print(
    f"Test samples affected           : "
    f"{len(vt_affected):,} / {len(test_final):,} "
    f"({100 * len(vt_affected) / len(test_final):.4f}%)"
)


# ------------------------------------------------------------
# 7. VERIFY RAW TITLE + IMAGE URL + ID AGAIN
# ------------------------------------------------------------

def nonempty_set(df, col):
    return set(
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    ) - {""}


print("\n" + "=" * 80)
print("FINAL EXACT CROSS-SPLIT VERIFICATION")
print("=" * 80)

for col in [ID_COL, RAW_TITLE_COL, URL_COL]:

    tr = nonempty_set(train_final, col)
    va = nonempty_set(val_final, col)
    te = nonempty_set(test_final, col)

    print(f"\n{col}:")
    print(f"  Train ↔ Val  : {len(tr & va):,}")
    print(f"  Train ↔ Test : {len(tr & te):,}")
    print(f"  Val ↔ Test   : {len(va & te):,}")


# ------------------------------------------------------------
# 8. SAVE OVERLAP DETAILS
# ------------------------------------------------------------

val_details.to_csv(
    "/kaggle/working/fakeddit_train_val_clean_title_overlap.csv",
    index=False
)

test_details.to_csv(
    "/kaggle/working/fakeddit_train_test_clean_title_overlap.csv",
    index=False
)

test_affected.to_csv(
    "/kaggle/working/fakeddit_test_samples_with_train_title_overlap.csv",
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("Saved:")
print("fakeddit_train_val_clean_title_overlap.csv")
print("fakeddit_train_test_clean_title_overlap.csv")
print("fakeddit_test_samples_with_train_title_overlap.csv")


Train ↔ Val clean_title OVERLAP
Unique overlapping clean_titles : 84
Val samples affected          : 86 / 12,400 (0.6935%)
Exact single-label matches       : 50 / 84 (59.52%)

TITLE LENGTH DISTRIBUTION
Titles <=  5 characters : 18 / 84 (21.43%)
Titles <= 10 characters : 41 / 84 (48.81%)
Titles <= 15 characters : 62 / 84 (73.81%)
Titles <= 20 characters : 70 / 84 (83.33%)
Titles <= 30 characters : 77 / 84 (91.67%)
Titles <= 50 characters : 81 / 84 (96.43%)

MOST FREQUENT OVERLAPPING clean_titles
--------------------------------------------------------------------------------
                   clean_title  title_length  train_count  val_count train_labels val_labels  exact_single_label_match
                         circa             5           11          1       [2, 4]        [2]                     False
                          dday             4            5          1       [2, 4]        [4]                     False
                       germany             7            4    